In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


We create one output folder where all audit files will be saved. This keeps your thesis cleaning work organized.

In [3]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

file_path = "/content/drive/MyDrive/IBS_7_Thesis/Python Files/DE_master_file_cleaned.csv"

output_folder = Path("/content/drive/MyDrive/IBS_7_Thesis/Python Files/cleaning_audit_outputs")
output_folder.mkdir(parents=True, exist_ok=True)

file_path

'/content/drive/MyDrive/IBS_7_Thesis/Python Files/DE_master_file_cleaned.csv'

We read every column as text. This is important because German-style numbers like 679.793,84 should not be wrongly converted by Pandas.

In [4]:
def load_csv_safely(path):
    encodings = ["utf-8-sig", "utf-8", "latin1"]

    last_error = None

    for enc in encodings:
        try:
            df = pd.read_csv(
                path,
                dtype="string",          # read everything as text first
                encoding=enc,
                low_memory=False,
                keep_default_na=False     # do not let pandas guess missing values automatically
            )
            print(f"File loaded successfully with encoding: {enc}")
            return df
        except Exception as e:
            last_error = e
            print(f"Could not load with encoding {enc}: {e}")

    raise last_error

df_raw = load_csv_safely(file_path)

print("Shape of raw data:", df_raw.shape)
df_raw.head()

File loaded successfully with encoding: utf-8-sig
Shape of raw data: (104099, 30)


,Name,Rechtsform,Land,PLZ,Ort,HR Amtsgericht,Register-ID,Status,North Data URL,USt.-Id.,...,Eigenkapital EUR,EK-Quote %,EK-Rendite %,Mitarbeiterzahl,Umsatz pro Mitarbeiter EUR,Steuern EUR,Kassenbestand EUR,Forderungen EUR,Verbindlichkeiten EUR,Personalaufwand EUR
0,Knut Rösch Baugrunduntersuchungen GmbH,GmbH,DE,22844.0,Norderstedt,Kiel,HRB 7129 KI,aktiv,https://www.northdata.de/Knut%20R%C3%B6sch%20B...,DE118666165,...,"200.063,46","29,43","-56,73",,,,"150.845,95","59.624,39","358.115,38",
1,Soulworks Developments GmbH,GmbH,DE,83646.0,Bad Tölz,München,HRB 288800,aktiv,https://www.northdata.de/Soulworks%20Developme...,DE260731679,...,"619.387,88","91,14","-35,09",,,,"333.451,04","331.284,83","29.415,29",
2,BEHR INGENIEURE GmbH,GmbH,DE,4103.0,Leipzig,Leipzig,HRB 11586,aktiv,https://www.northdata.de/BEHR%20INGENIEURE%20G...,DE177768304,...,"156.773,48","23,07","85,04",12.0,"91.666,67",,"425.336,85","181.877,25","162.438,7",
3,GLASFAKTOR Ingenieure GmbH,GmbH,DE,1324.0,Dresden,Dresden,HRB 30711,aktiv,https://www.northdata.de/GLASFAKTOR%20Ingenieu...,DE280122771,...,"501.018,27","73,73","43,11",,,,"619.318,74","49.954,88","136.972,16",
4,Novum Analytik GmbH,GmbH,DE,74078.0,Heilbronn,Stuttgart,HRB 723862,aktiv,https://www.northdata.de/Novum%20Analytik%20Gm...,DE255820468,...,"534.165,84","78,62","1,22",13.0,"246.153,85",,"162.635,85","191.789,74","112.454,35",


In [5]:
print("Number of columns:", len(df_raw.columns))

for i, col in enumerate(df_raw.columns, start=1):
    print(f"{i}. {col}")

Number of columns: 30
1. Name
2. Rechtsform
3. Land
4. PLZ
5. Ort
6. HR Amtsgericht
7. Register-ID
8. Status
9. North Data URL
10. USt.-Id.
11. Branche (WZ)
12. Gegenstand
13. Finanzkennzahlen Datum
14. Stamm-/Grundkapital EUR
15. Bilanzsumme EUR
16. Gewinn EUR
17. Gewinn CAGR %
18. Umsatz EUR
19. Umsatz CAGR %
20. Umsatzrendite %
21. Eigenkapital EUR
22. EK-Quote %
23. EK-Rendite %
24. Mitarbeiterzahl
25. Umsatz pro Mitarbeiter EUR
26. Steuern EUR
27. Kassenbestand EUR
28. Forderungen EUR
29. Verbindlichkeiten EUR
30. Personalaufwand EUR


**Cell 5 — Create a working copy only for audit**

In [6]:
df_audit = df_raw.copy()

# Convert empty-looking strings into missing values only for audit
missing_tokens = ["", " ", "nan", "NaN", "NULL", "null", "None", "NA", "N/A", "n/a", "-", "--"]

for col in df_audit.columns:
    df_audit[col] = df_audit[col].astype("string").str.strip()
    df_audit[col] = df_audit[col].replace(missing_tokens, pd.NA)

print("Raw data shape:", df_raw.shape)
print("Audit copy shape:", df_audit.shape)

Raw data shape: (104099, 30)
Audit copy shape: (104099, 30)


**Cell 6 — Basic missing-value audit for every column**

This tells us which columns are useful and which columns are too empty for thesis analysis.

In [7]:
column_audit = []

for col in df_audit.columns:
    total = len(df_audit)
    missing = df_audit[col].isna().sum()
    non_missing = total - missing
    unique_values = df_audit[col].nunique(dropna=True)

    sample_values = (
        df_audit[col]
        .dropna()
        .astype(str)
        .drop_duplicates()
        .head(8)
        .tolist()
    )

    column_audit.append({
        "column": col,
        "total_rows": total,
        "non_missing": non_missing,
        "missing": missing,
        "missing_pct": round(missing / total * 100, 2),
        "unique_values": unique_values,
        "sample_values": sample_values
    })

column_audit_df = pd.DataFrame(column_audit)

column_audit_df.sort_values("missing_pct", ascending=False)

,column,total_rows,non_missing,missing,missing_pct,unique_values,sample_values
25,Steuern EUR,104099,1326,102773,98.73,1140,"[10.660,25, 92.533,01, 5.089,82, 315,41, 717,3..."
29,Personalaufwand EUR,104099,8572,95527,91.77,7475,"[234.234,92, 7.289,36, 348.215,74, 1.211.713,0..."
24,Umsatz pro Mitarbeiter EUR,104099,32749,71350,68.54,8553,"[91.666,67, 246.153,85, 150.000, 220.000, 122...."
23,Mitarbeiterzahl,104099,32754,71345,68.54,947,"[12.0, 13.0, 5.0, 9.0, 4.0, 1.0, 20.0, 18.0]"
16,Gewinn CAGR %,104099,42981,61118,58.71,19875,"[0,62, 12,29, 38,46, 75,62, 133,18, -59,72, -9..."
9,USt.-Id.,104099,50638,53461,51.36,46193,"[DE118666165, DE260731679, DE177768304, DE2801..."
22,EK-Rendite %,104099,76660,27439,26.36,17778,"[-56,73, -35,09, 85,04, 43,11, 1,22, 0, 23,07,..."
19,Umsatzrendite %,104099,84363,19736,18.96,9581,"[-16,45, -5,05, 12,12, 19,64, 0,20, 0, 5,32, 8..."
15,Gewinn EUR,104099,84580,19519,18.75,67287,"[-113.496,26, -217.337,80, 133.323,1, 216.008,..."
26,Kassenbestand EUR,104099,84773,19326,18.57,77594,"[150.845,95, 333.451,04, 425.336,85, 619.318,7..."


**Cell 7 — Save missing-value audit**

In [8]:
column_audit_path = output_folder / "part1_column_missing_audit.csv"

column_audit_df.to_csv(column_audit_path, index=False, encoding="utf-8-sig")

print("Saved column missing audit to:")
print(column_audit_path)

Saved column missing audit to:
/content/drive/MyDrive/IBS_7_Thesis/Python Files/cleaning_audit_outputs/part1_column_missing_audit.csv


**Cell 8 — Define a function to classify value types**

In [9]:
def classify_value_type(value):
    if pd.isna(value):
        return "missing"

    x = str(value).strip()

    if x == "":
        return "missing"

    # URL
    if re.search(r"https?://|www\.", x, flags=re.IGNORECASE):
        return "url"

    # Email
    if re.search(r"^[\w\.-]+@[\w\.-]+\.\w+$", x):
        return "email"

    # German-style date, for example 31.12.2023
    if re.search(r"^\d{1,2}\.\d{1,2}\.\d{2,4}$", x):
        return "date_like"

    # Contains currency/unit
    if re.search(r"EUR|€|DEM|\$|Mio\.?|Mrd\.?", x, flags=re.IGNORECASE):
        return "currency_or_unit_number"

    # Percentage
    if "%" in x:
        return "percentage"

    # Pure number-like value with separators, minus signs, commas, dots
    if re.search(r"^-?[\d\.,]+$", x):
        return "numeric_like"

    # Contains both letters and numbers
    if re.search(r"[A-Za-zÄÖÜäöüß]", x) and re.search(r"\d", x):
        return "mixed_text_number"

    # Text only
    if re.search(r"[A-Za-zÄÖÜäöüß]", x):
        return "text"

    return "other"

In [10]:
type_audit_rows = []

for col in df_audit.columns:
    value_types = df_audit[col].apply(classify_value_type)
    type_counts = value_types.value_counts(dropna=False).to_dict()

    row = {"column": col}
    row.update(type_counts)
    type_audit_rows.append(row)

type_audit_df = pd.DataFrame(type_audit_rows).fillna(0)

# Put column name first, then type columns
type_cols = [c for c in type_audit_df.columns if c != "column"]
type_audit_df[type_cols] = type_audit_df[type_cols].astype(int)

type_audit_df

,column,text,currency_or_unit_number,mixed_text_number,url,percentage,missing,numeric_like,other,date_like
0,Name,96835,3914,3346,2,2,0,0,0,0
1,Rechtsform,104075,0,0,0,0,24,0,0,0
2,Land,104099,0,0,0,0,0,0,0,0
3,PLZ,0,0,0,0,0,73,104026,0,0
4,Ort,103795,304,0,0,0,0,0,0,0
5,HR Amtsgericht,103620,426,0,0,0,53,0,0,0
6,Register-ID,0,0,104049,0,0,13,37,0,0
7,Status,104099,0,0,0,0,0,0,0,0
8,North Data URL,0,0,0,104099,0,0,0,0,0
9,USt.-Id.,0,0,50638,0,0,53461,0,0,0


In [11]:
type_audit_path = output_folder / "part1_column_type_audit.csv"

type_audit_df.to_csv(type_audit_path, index=False, encoding="utf-8-sig")

print("Saved column type audit to:")
print(type_audit_path)

Saved column type audit to:
/content/drive/MyDrive/IBS_7_Thesis/Python Files/cleaning_audit_outputs/part1_column_type_audit.csv


**Part 1B — Specific audit for financial/numeric columns**

**Now we check the columns needed for your ratio construction.**

In [12]:
important_numeric_cols = [
    "Stamm-/Grundkapital EUR",
    "Bilanzsumme EUR",
    "Gewinn EUR",
    "Gewinn CAGR %",
    "Umsatz EUR",
    "Umsatz CAGR %",
    "Umsatzrendite %",
    "Eigenkapital EUR",
    "EK-Quote %",
    "EK-Rendite %",
    "Mitarbeiterzahl",
    "Umsatz pro Mitarbeiter EUR",
    "Steuern EUR",
    "Kassenbestand EUR",
    "Forderungen EUR",
    "Verbindlichkeiten EUR",
    "Personalaufwand EUR"
]

existing_numeric_cols = [col for col in important_numeric_cols if col in df_audit.columns]
missing_numeric_cols = [col for col in important_numeric_cols if col not in df_audit.columns]

print("Existing important numeric columns:")
print(existing_numeric_cols)

print("\nMissing expected numeric columns:")
print(missing_numeric_cols)

Existing important numeric columns:
['Stamm-/Grundkapital EUR', 'Bilanzsumme EUR', 'Gewinn EUR', 'Gewinn CAGR %', 'Umsatz EUR', 'Umsatz CAGR %', 'Umsatzrendite %', 'Eigenkapital EUR', 'EK-Quote %', 'EK-Rendite %', 'Mitarbeiterzahl', 'Umsatz pro Mitarbeiter EUR', 'Steuern EUR', 'Kassenbestand EUR', 'Forderungen EUR', 'Verbindlichkeiten EUR', 'Personalaufwand EUR']

Missing expected numeric columns:
[]


**Cell 12 — Audit number formats inside financial columns**

In [13]:
def audit_numeric_format(series):
    s = series.dropna().astype(str).str.strip()

    return {
        "non_missing": len(s),
        "contains_comma": s.str.contains(",", regex=False).sum(),
        "contains_dot": s.str.contains(".", regex=False).sum(),
        "contains_both_dot_and_comma": s.str.contains(r"\.", regex=True).astype(int).mul(
            s.str.contains(",", regex=False).astype(int)
        ).sum(),
        "contains_minus": s.str.contains("-", regex=False).sum(),
        "contains_question_mark": s.str.contains(r"\?", regex=True).sum(),
        "contains_percent": s.str.contains("%", regex=False).sum(),
        "contains_eur": s.str.contains("EUR|€", case=False, regex=True).sum(),
        "contains_mio": s.str.contains(r"\bMio\.?\b", case=False, regex=True).sum(),
        "contains_mrd": s.str.contains(r"\bMrd\.?\b", case=False, regex=True).sum(),
        "contains_dem": s.str.contains("DEM", case=False, regex=True).sum(),
        "contains_dollar": s.str.contains(r"\$", regex=True).sum(),
        "contains_letters": s.str.contains(r"[A-Za-zÄÖÜäöüß]", regex=True).sum(),
        "contains_spaces": s.str.contains(r"\s", regex=True).sum(),
        "sample_values": s.drop_duplicates().head(12).tolist()
    }

numeric_format_audit = []

for col in existing_numeric_cols:
    row = {"column": col}
    row.update(audit_numeric_format(df_audit[col]))
    numeric_format_audit.append(row)

numeric_format_audit_df = pd.DataFrame(numeric_format_audit)

numeric_format_audit_df

,column,non_missing,contains_comma,contains_dot,contains_both_dot_and_comma,contains_minus,contains_question_mark,contains_percent,contains_eur,contains_mio,contains_mrd,contains_dem,contains_dollar,contains_letters,contains_spaces,sample_values
0,Stamm-/Grundkapital EUR,99603,8300,97988,7856,0,0,0,13250,81,2,926,0,14177,14177,"[26.000, 25.000, 25.600, 118.000, 200.000, 52...."
1,Bilanzsumme EUR,98862,84902,98659,84759,0,0,0,14196,2392,0,0,1,14196,14197,"[679.793,84, 679.593,17, 679.558,8, 679.529,43..."
2,Gewinn EUR,84580,66002,71068,62732,15414,3847,0,11301,149,0,0,1,11301,11302,"[-113.496,26, -217.337,80, 133.323,1, 216.008,..."
3,Gewinn CAGR %,42981,42492,736,731,14518,1923,4401,0,0,0,0,0,0,4401,"[0,62, 12,29, 38,46, 75,62, 133,18, -59,72, -9..."
4,Umsatz EUR,98853,4384,98532,4260,3,1,0,14197,3086,4,0,1,14197,14198,"[690.000, 4.300.000, 1.100.000, 3.200.000, 650..."
5,Umsatz CAGR %,94990,88300,583,489,25876,4190,13182,0,0,0,0,0,0,13182,"[-4,80, 7,80, -4,09, 4,85, -2,22, -9,86, 2,53,..."
6,Umsatzrendite %,84363,71595,233,232,15357,3823,11227,0,0,0,0,0,0,11227,"[-16,45, -5,05, 12,12, 19,64, 0,20, 0, 5,32, 8..."
7,Eigenkapital EUR,98025,79210,95822,78426,6370,1631,0,14047,1264,2,0,1,14047,14048,"[200.063,46, 619.387,88, 156.773,48, 501.018,2..."
8,EK-Quote %,97982,94489,51,50,6355,1630,14044,0,0,0,0,0,0,14044,"[29,43, 91,14, 23,07, 73,73, 78,62, 29,86, 71,..."
9,EK-Rendite %,76660,64870,567,556,11846,2774,9774,0,0,0,0,0,0,9774,"[-56,73, -35,09, 85,04, 43,11, 1,22, 0, 23,07,..."


In [14]:
numeric_format_audit_path = output_folder / "part1_numeric_format_audit.csv"

numeric_format_audit_df.to_csv(numeric_format_audit_path, index=False, encoding="utf-8-sig")

print("Saved numeric format audit to:")
print(numeric_format_audit_path)

Saved numeric format audit to:
/content/drive/MyDrive/IBS_7_Thesis/Python Files/cleaning_audit_outputs/part1_numeric_format_audit.csv


**Part 1C — Show suspicious examples**

**Cell 14 — Show strange values in important financial columns**

In [15]:
def show_suspicious_values(df, col, n=20):
    s = df[col].dropna().astype(str).str.strip()

    suspicious = s[
        s.str.contains(r"\?|DEM|\$|Mio|Mrd|[A-Za-zÄÖÜäöüß]", case=False, regex=True)
    ].drop_duplicates()

    print(f"\nColumn: {col}")
    print(f"Suspicious unique values found: {len(suspicious)}")
    display(suspicious.head(n).to_frame(name="suspicious_values"))

for col in existing_numeric_cols:
    show_suspicious_values(df_audit, col, n=20)


Column: Stamm-/Grundkapital EUR
Suspicious unique values found: 1174


,suspicious_values
5405,61.380 EUR
5406,25.000 EUR
5408,840.000 EUR
5409,109.600 EUR
5410,30.000 EUR
5416,25.300 EUR
5418,512.000 EUR
5419,50.000 EUR
5420,50.000 DEM
5421,48.775 EUR



Column: Bilanzsumme EUR
Suspicious unique values found: 11022


,suspicious_values
5405,1.479.022 EUR
5406,1.478.985 EUR
5407,1.478.446 EUR
5408,1.476.890 EUR
5409,1.476.532 EUR
5410,1.476.290 EUR
5411,1.475.709 EUR
5412,1.474.449 EUR
5413,1.474.109 EUR
5414,1.473.993 EUR



Column: Gewinn EUR
Suspicious unique values found: 8684


...

,suspicious_values
5405,821.223 EUR
5406,121.273 EUR
5408,"152,11 EUR"
5410,499.107 EUR
5411,108.887 EUR
5412,781.264 EUR
5413,47.977 EUR
5414,761.352 EUR
5415,"7.264,20 EUR"
5416,744.801 EUR



Column: Verbindlichkeiten EUR
Suspicious unique values found: 10953


,suspicious_values
5405,496.969 EUR
5406,1.427.211 EUR
5407,"9.675,79 EUR"
5408,216.196 EUR
5409,"10.812,48 EUR"
5410,"119,00 EUR"
5411,1.429.933 EUR
5412,1.474.449 EUR
5413,1.450.513 EUR
5414,231.680 EUR



Column: Personalaufwand EUR
Suspicious unique values found: 785


,suspicious_values
5420,1.679.295 EUR
5443,1.331.375 EUR
5513,1.141.287 EUR
5515,1.557.703 EUR
5610,"0,00 EUR"
5615,"20,2 Mio. EUR"
5633,70.043 EUR
5635,961.805 EUR
5654,1.285.950 EUR
5679,1.126.199 EUR


**Cell 15 — Check important categorical columns**

In [16]:
important_categorical_cols = [
    "Land",
    "Status",
    "Branche (WZ)",
    "Finanzkennzahlen Datum",
    "Register-ID",
    "North Data URL",
    "Name"
]

existing_categorical_cols = [col for col in important_categorical_cols if col in df_audit.columns]

for col in existing_categorical_cols:
    print("\n" + "="*80)
    print(f"Column: {col}")
    print("Missing:", df_audit[col].isna().sum())
    print("Unique values:", df_audit[col].nunique(dropna=True))
    display(df_audit[col].value_counts(dropna=False).head(20).to_frame("count"))


Column: Land
Missing: 0
Unique values: 5


,count
Land,
DE,104093
CH,2
AT,2
IT,1
BE,1



Column: Status
Missing: 0
Unique values: 5


,count
Status,
aktiv,62848
active,37031
erloschen,2335
in Liquidation,1055
liquidation,830



Column: Branche (WZ)
Missing: 49
Unique values: 646


...

,count
Register-ID,
<NA>,13
HRB 12433,10
HRB 2442,9
HRB 6564,9
HRB 2829,9
HRB 12626,9
HRB 5973,9
HRB 5749,9
HRB 12146,8



Column: North Data URL
Missing: 0
Unique values: 96227


,count
North Data URL,
"https://www.northdata.de/LR%20Verm%C3%B6gensverwaltungs%20GmbH,%20N%C3%BCrnberg/HRB%2029632",4
"https://www.northdata.de/bioM%C3%A9rieux%20Deutschland%20GmbH,%20N%C3%BCrtingen/Amtsgericht%20Stuttgart%20HRB%20220743",4
"https://www.northdata.de/J%C2%B7&%20W%C2%B7%20M%C3%BCller%20GmbH%20&%20Co%C2%B7%20KG,%20Leverkusen/Amtsgericht%20K%C3%B6ln%20HRA%2023865",4
"https://www.northdata.de/AEROSPACE%20COMPOSITES%20GmbH,%20Laupheim/Amtsgericht%20Ulm%20HRB%20733388",3
"https://www.northdata.de/Wolf%20Anlagen-Technik%20GmbH%20&%20Co%C2%B7%20KG,%20Geisenfeld/Amtsgericht%20Ingolstadt%20HRA%20170353",3
"https://www.northdata.de/LOHMANN%20&%20Co%C2%B7%20AG,%20M%C3%B6ckern/Amtsgericht%20Oldenburg%20HRB%20110760",3
"https://www.northdata.de/GBE%20Deutschland%20GmbH%20&%20Co%C2%B7%20KG,%20Hamburg/HRA%20132462",3
"https://www.northdata.de/Schwarzer%20Verm%C3%B6gensverwaltungs%20GmbH,%20Berlin/Amtsgericht%20Charlottenburg%20(Berlin)%20HRB%2095930%20B",3
"https://www.northdata.de/Heiner%20Haubrich%20GmbH,%20D%C3%BCsseldorf/HRB%2069490",3



Column: Name
Missing: 0
Unique values: 95604


,count
Name,
pro optik Augenoptik Fachgeschäft GmbH,47
BüchnerBarella Versicherungsmakler GmbH,12
Fischer GmbH,9
Volksbank eG,8
Autohaus Müller GmbH,7
FC-Planung GmbH,6
Hofmann GmbH,5
Braun GmbH,5
Auto Bierschneider GmbH,5


**Cell 16 - Check duplicate situation**

In [17]:
print("Exact duplicate rows:", df_audit.duplicated().sum())

if "Register-ID" in df_audit.columns:
    print("Duplicate Register-ID rows:", df_audit.duplicated(subset=["Register-ID"], keep=False).sum())

if "North Data URL" in df_audit.columns:
    print("Duplicate North Data URL rows:", df_audit.duplicated(subset=["North Data URL"], keep=False).sum())

if "Name" in df_audit.columns:
    print("Duplicate Name rows:", df_audit.duplicated(subset=["Name"], keep=False).sum())

Exact duplicate rows: 6510
Duplicate Register-ID rows: 44610
Duplicate North Data URL rows: 15609
Duplicate Name rows: 16608


**Cell 17 - Financial date Audit**

In [18]:
if "Finanzkennzahlen Datum" in df_audit.columns:
    date_test = pd.to_datetime(
        df_audit["Finanzkennzahlen Datum"],
        format="%d.%m.%Y",
        errors="coerce"
    )

    financial_year = date_test.dt.year

    date_audit = pd.DataFrame({
        "original_date": df_audit["Finanzkennzahlen Datum"],
        "parsed_date": date_test,
        "financial_year": financial_year
    })

    print("Rows with valid financial date:", date_test.notna().sum())
    print("Rows with invalid/missing financial date:", date_test.isna().sum())

    display(financial_year.value_counts(dropna=False).sort_index().to_frame("count"))

Rows with valid financial date: 98912
Rows with invalid/missing financial date: 5187


,count
Finanzkennzahlen Datum,
2006.0,5
2007.0,4
2008.0,4
2009.0,11
2010.0,27
2011.0,162
2012.0,95
2013.0,40
2014.0,53


In [19]:
print("PART 1 SUMMARY")
print("="*60)
print("Raw shape:", df_raw.shape)
print("Audit copy shape:", df_audit.shape)
print("Number of columns:", len(df_raw.columns))
print("Column audit saved:", column_audit_path)
print("Type audit saved:", type_audit_path)
print("Numeric format audit saved:", numeric_format_audit_path)
print("="*60)

print("\nNext step will be Part 2:")
print("Clean numeric columns using a robust German-number parser.")

PART 1 SUMMARY
Raw shape: (104099, 30)
Audit copy shape: (104099, 30)
Number of columns: 30
Column audit saved: /content/drive/MyDrive/IBS_7_Thesis/Python Files/cleaning_audit_outputs/part1_column_missing_audit.csv
Type audit saved: /content/drive/MyDrive/IBS_7_Thesis/Python Files/cleaning_audit_outputs/part1_column_type_audit.csv
Numeric format audit saved: /content/drive/MyDrive/IBS_7_Thesis/Python Files/cleaning_audit_outputs/part1_numeric_format_audit.csv

Next step will be Part 2:
Clean numeric columns using a robust German-number parser.


In [20]:
df_raw.shape


(104099, 30)

In [21]:
column_audit_df.sort_values("missing_pct", ascending=False).head(15)

,column,total_rows,non_missing,missing,missing_pct,unique_values,sample_values
25,Steuern EUR,104099,1326,102773,98.73,1140,"[10.660,25, 92.533,01, 5.089,82, 315,41, 717,3..."
29,Personalaufwand EUR,104099,8572,95527,91.77,7475,"[234.234,92, 7.289,36, 348.215,74, 1.211.713,0..."
24,Umsatz pro Mitarbeiter EUR,104099,32749,71350,68.54,8553,"[91.666,67, 246.153,85, 150.000, 220.000, 122...."
23,Mitarbeiterzahl,104099,32754,71345,68.54,947,"[12.0, 13.0, 5.0, 9.0, 4.0, 1.0, 20.0, 18.0]"
16,Gewinn CAGR %,104099,42981,61118,58.71,19875,"[0,62, 12,29, 38,46, 75,62, 133,18, -59,72, -9..."
9,USt.-Id.,104099,50638,53461,51.36,46193,"[DE118666165, DE260731679, DE177768304, DE2801..."
22,EK-Rendite %,104099,76660,27439,26.36,17778,"[-56,73, -35,09, 85,04, 43,11, 1,22, 0, 23,07,..."
19,Umsatzrendite %,104099,84363,19736,18.96,9581,"[-16,45, -5,05, 12,12, 19,64, 0,20, 0, 5,32, 8..."
15,Gewinn EUR,104099,84580,19519,18.75,67287,"[-113.496,26, -217.337,80, 133.323,1, 216.008,..."
26,Kassenbestand EUR,104099,84773,19326,18.57,77594,"[150.845,95, 333.451,04, 425.336,85, 619.318,7..."


In [22]:
numeric_format_audit_df

,column,non_missing,contains_comma,contains_dot,contains_both_dot_and_comma,contains_minus,contains_question_mark,contains_percent,contains_eur,contains_mio,contains_mrd,contains_dem,contains_dollar,contains_letters,contains_spaces,sample_values
0,Stamm-/Grundkapital EUR,99603,8300,97988,7856,0,0,0,13250,81,2,926,0,14177,14177,"[26.000, 25.000, 25.600, 118.000, 200.000, 52...."
1,Bilanzsumme EUR,98862,84902,98659,84759,0,0,0,14196,2392,0,0,1,14196,14197,"[679.793,84, 679.593,17, 679.558,8, 679.529,43..."
2,Gewinn EUR,84580,66002,71068,62732,15414,3847,0,11301,149,0,0,1,11301,11302,"[-113.496,26, -217.337,80, 133.323,1, 216.008,..."
3,Gewinn CAGR %,42981,42492,736,731,14518,1923,4401,0,0,0,0,0,0,4401,"[0,62, 12,29, 38,46, 75,62, 133,18, -59,72, -9..."
4,Umsatz EUR,98853,4384,98532,4260,3,1,0,14197,3086,4,0,1,14197,14198,"[690.000, 4.300.000, 1.100.000, 3.200.000, 650..."
5,Umsatz CAGR %,94990,88300,583,489,25876,4190,13182,0,0,0,0,0,0,13182,"[-4,80, 7,80, -4,09, 4,85, -2,22, -9,86, 2,53,..."
6,Umsatzrendite %,84363,71595,233,232,15357,3823,11227,0,0,0,0,0,0,11227,"[-16,45, -5,05, 12,12, 19,64, 0,20, 0, 5,32, 8..."
7,Eigenkapital EUR,98025,79210,95822,78426,6370,1631,0,14047,1264,2,0,1,14047,14048,"[200.063,46, 619.387,88, 156.773,48, 501.018,2..."
8,EK-Quote %,97982,94489,51,50,6355,1630,14044,0,0,0,0,0,0,14044,"[29,43, 91,14, 23,07, 73,73, 78,62, 29,86, 71,..."
9,EK-Rendite %,76660,64870,567,556,11846,2774,9774,0,0,0,0,0,0,9774,"[-56,73, -35,09, 85,04, 43,11, 1,22, 0, 23,07,..."


## **Part 2 — Clean numeric columns with robust German-number parser**

**cell 19 — Create numeric-cleaning copy**

In [23]:
df_num = df_audit.copy()

print("Input shape for numeric cleaning:", df_num.shape)

Input shape for numeric cleaning: (104099, 30)


**Cell 20 — Define original-to-clean column mapping**

In [24]:
numeric_column_map = {
    "Stamm-/Grundkapital EUR": "share_capital",
    "Bilanzsumme EUR": "assets",
    "Gewinn EUR": "profit",
    "Gewinn CAGR %": "profit_cagr_pct",
    "Umsatz EUR": "revenue",
    "Umsatz CAGR %": "revenue_cagr_pct",
    "Umsatzrendite %": "profit_margin_original_pct",
    "Eigenkapital EUR": "equity",
    "EK-Quote %": "equity_ratio_original_pct",
    "EK-Rendite %": "roe_original_pct",
    "Mitarbeiterzahl": "employees",
    "Umsatz pro Mitarbeiter EUR": "revenue_per_employee_original",
    "Steuern EUR": "taxes",
    "Kassenbestand EUR": "cash",
    "Forderungen EUR": "receivables",
    "Verbindlichkeiten EUR": "liabilities",
    "Personalaufwand EUR": "personnel_expense"
}

numeric_column_map

{'Stamm-/Grundkapital EUR': 'share_capital',
 'Bilanzsumme EUR': 'assets',
 'Gewinn EUR': 'profit',
 'Gewinn CAGR %': 'profit_cagr_pct',
 'Umsatz EUR': 'revenue',
 'Umsatz CAGR %': 'revenue_cagr_pct',
 'Umsatzrendite %': 'profit_margin_original_pct',
 'Eigenkapital EUR': 'equity',
 'EK-Quote %': 'equity_ratio_original_pct',
 'EK-Rendite %': 'roe_original_pct',
 'Mitarbeiterzahl': 'employees',
 'Umsatz pro Mitarbeiter EUR': 'revenue_per_employee_original',
 'Steuern EUR': 'taxes',
 'Kassenbestand EUR': 'cash',
 'Forderungen EUR': 'receivables',
 'Verbindlichkeiten EUR': 'liabilities',
 'Personalaufwand EUR': 'personnel_expense'}

**Cell 21 - Create robust German number parser**

In [25]:
import re
import numpy as np
import pandas as pd

def normalize_missing_value(value):
    if pd.isna(value):
        return None

    x = str(value).strip()

    missing_tokens = [
        "", " ", "nan", "NaN", "NULL", "null",
        "None", "NA", "N/A", "n/a", "-", "--"
    ]

    if x in missing_tokens:
        return None

    return x


def parse_german_number(value):
    original = value
    x = normalize_missing_value(value)

    result = {
        "value": np.nan,
        "status": "missing",
        "has_uncertain_marker": False,
        "has_percent": False,
        "has_mio": False,
        "has_mrd": False,
        "has_non_eur_currency": False,
        "original": original
    }

    if x is None:
        return result

    result["status"] = "invalid"

    # Standardize minus signs and spaces
    x = x.replace("\u2212", "-").replace("\xa0", " ").strip()

    # Flags
    result["has_uncertain_marker"] = "?" in x
    result["has_percent"] = "%" in x
    result["has_mio"] = bool(re.search(r"\bMio\.?\b", x, flags=re.IGNORECASE))
    result["has_mrd"] = bool(re.search(r"\bMrd\.?\b", x, flags=re.IGNORECASE))
    result["has_non_eur_currency"] = bool(
        re.search(r"\bDEM\b|\$", x, flags=re.IGNORECASE)
    )

    # Non-EUR values are not comparable for thesis financial analysis
    if result["has_non_eur_currency"]:
        result["status"] = "non_eur_currency"
        return result

    # Unit multiplier
    multiplier = 1.0

    if result["has_mio"]:
        multiplier = 1_000_000.0

    if result["has_mrd"]:
        multiplier = 1_000_000_000.0

    # Remove symbols, units and uncertainty signs
    y = x
    y = re.sub(r"\?", "", y)
    y = re.sub(r"EUR|€|%", "", y, flags=re.IGNORECASE)
    y = re.sub(r"\bMio\.?\b|\bMrd\.?\b", "", y, flags=re.IGNORECASE)
    y = y.strip()

    # Handle accounting-style negatives if they exist, e.g. (1.234,56)
    if re.fullmatch(r"\(.*\)", y):
        y = "-" + y.strip("()")

    # Remove spaces and apostrophe separators
    y = re.sub(r"[\s']", "", y)

    # Only digits, comma, dot, minus and plus should remain
    if not re.fullmatch(r"[+-]?[\d\.,]+", y):
        result["status"] = "invalid_format"
        return result

    # German style: 1.234,56 -> 1234.56
    if "," in y and "." in y:
        y_clean = y.replace(".", "").replace(",", ".")

    # German decimal comma: 123,45 -> 123.45
    elif "," in y:
        y_clean = y.replace(",", ".")

    # Dot can mean thousands or decimal
    elif "." in y:
        # 1.234 or 1.234.567 means thousands separator
        if re.fullmatch(r"[+-]?\d{1,3}(\.\d{3})+", y):
            y_clean = y.replace(".", "")
        else:
            # 12.0 remains 12.0
            y_clean = y

    else:
        y_clean = y

    try:
        result["value"] = float(y_clean) * multiplier
        result["status"] = "converted"
        return result

    except Exception:
        result["status"] = "conversion_error"
        return result

**Cell 22 - Test parser on known difficult examples**

In [26]:
test_values = [
    "679.793,84",
    "690.000",
    "-113.496,26",
    "0,62",
    "25.000 EUR",
    "?78,6 %",
    "11 Mio. EUR",
    "1,2 Mrd. EUR",
    "50.000 DEM",
    "257,33 $",
    "12.0",
    "1.887",
    "7,5",
    "474,2 Mio. EUR",
    "?14,3 Mio. EUR"
]

for val in test_values:
    parsed = parse_german_number(val)
    print(val, "→", parsed["value"], "| status:", parsed["status"])

679.793,84 → 679793.84 | status: converted
690.000 → 690000.0 | status: converted
-113.496,26 → -113496.26 | status: converted
0,62 → 0.62 | status: converted
25.000 EUR → 25000.0 | status: converted
?78,6 % → 78.6 | status: converted
11 Mio. EUR → 11000000.0 | status: converted
1,2 Mrd. EUR → 1200000000.0 | status: converted
50.000 DEM → nan | status: non_eur_currency
257,33 $ → nan | status: non_eur_currency
12.0 → 12.0 | status: converted
1.887 → 1887.0 | status: converted
7,5 → 7.5 | status: converted
474,2 Mio. EUR → 474200000.0 | status: converted
?14,3 Mio. EUR → 14300000.0 | status: converted


**Cell 23 - Apply parser to all important numeric columns**

In [27]:
for raw_col, clean_col in numeric_column_map.items():

    print(f"Converting: {raw_col}  →  {clean_col}")

    parsed_series = df_num[raw_col].apply(parse_german_number)

    df_num[clean_col] = parsed_series.apply(lambda x: x["value"])
    df_num[clean_col + "_parse_status"] = parsed_series.apply(lambda x: x["status"])
    df_num[clean_col + "_uncertain"] = parsed_series.apply(lambda x: x["has_uncertain_marker"])
    df_num[clean_col + "_has_percent"] = parsed_series.apply(lambda x: x["has_percent"])
    df_num[clean_col + "_has_mio"] = parsed_series.apply(lambda x: x["has_mio"])
    df_num[clean_col + "_has_mrd"] = parsed_series.apply(lambda x: x["has_mrd"])
    df_num[clean_col + "_non_eur_currency"] = parsed_series.apply(lambda x: x["has_non_eur_currency"])

print("Numeric conversion completed.")
print("Shape after adding clean numeric columns:", df_num.shape)

Converting: Stamm-/Grundkapital EUR  →  share_capital
Converting: Bilanzsumme EUR  →  assets
Converting: Gewinn EUR  →  profit
Converting: Gewinn CAGR %  →  profit_cagr_pct
Converting: Umsatz EUR  →  revenue
Converting: Umsatz CAGR %  →  revenue_cagr_pct
Converting: Umsatzrendite %  →  profit_margin_original_pct
Converting: Eigenkapital EUR  →  equity
Converting: EK-Quote %  →  equity_ratio_original_pct
Converting: EK-Rendite %  →  roe_original_pct
Converting: Mitarbeiterzahl  →  employees
Converting: Umsatz pro Mitarbeiter EUR  →  revenue_per_employee_original
Converting: Steuern EUR  →  taxes
Converting: Kassenbestand EUR  →  cash
Converting: Forderungen EUR  →  receivables


/tmp/ipykernel_8591/3019696567.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_num[clean_col + "_uncertain"] = parsed_series.apply(lambda x: x["has_uncertain_marker"])
/tmp/ipykernel_8591/3019696567.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_num[clean_col + "_has_percent"] = parsed_series.apply(lambda x: x["has_percent"])
/tmp/ipykernel_8591/3019696567.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consi

Converting: Verbindlichkeiten EUR  →  liabilities


/tmp/ipykernel_8591/3019696567.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_num[clean_col] = parsed_series.apply(lambda x: x["value"])
/tmp/ipykernel_8591/3019696567.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_num[clean_col + "_parse_status"] = parsed_series.apply(lambda x: x["status"])
/tmp/ipykernel_8591/3019696567.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once usin

Converting: Personalaufwand EUR  →  personnel_expense
Numeric conversion completed.
Shape after adding clean numeric columns: (104099, 149)


/tmp/ipykernel_8591/3019696567.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_num[clean_col] = parsed_series.apply(lambda x: x["value"])
/tmp/ipykernel_8591/3019696567.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_num[clean_col + "_parse_status"] = parsed_series.apply(lambda x: x["status"])
/tmp/ipykernel_8591/3019696567.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once usin

**Cell 24 - Create numeric conversion audit table**

In [28]:
conversion_audit_rows = []

for raw_col, clean_col in numeric_column_map.items():

    original_non_missing = df_num[raw_col].notna().sum()
    converted_count = df_num[clean_col].notna().sum()

    status_counts = (
        df_num[clean_col + "_parse_status"]
        .value_counts(dropna=False)
        .to_dict()
    )

    row = {
        "original_column": raw_col,
        "clean_column": clean_col,
        "original_non_missing": original_non_missing,
        "converted_count": converted_count,
        "missing_after_conversion": df_num[clean_col].isna().sum(),
        "converted": status_counts.get("converted", 0),
        "missing": status_counts.get("missing", 0),
        "non_eur_currency": status_counts.get("non_eur_currency", 0),
        "invalid_format": status_counts.get("invalid_format", 0),
        "conversion_error": status_counts.get("conversion_error", 0),
        "uncertain_marker_count": df_num[clean_col + "_uncertain"].sum(),
        "mio_count": df_num[clean_col + "_has_mio"].sum(),
        "mrd_count": df_num[clean_col + "_has_mrd"].sum(),
        "negative_count": (df_num[clean_col] < 0).sum(),
        "zero_count": (df_num[clean_col] == 0).sum(),
        "min": df_num[clean_col].min(skipna=True),
        "median": df_num[clean_col].median(skipna=True),
        "max": df_num[clean_col].max(skipna=True)
    }

    conversion_audit_rows.append(row)

conversion_audit_df = pd.DataFrame(conversion_audit_rows)

conversion_audit_df

,original_column,clean_column,original_non_missing,converted_count,missing_after_conversion,converted,missing,non_eur_currency,invalid_format,conversion_error,uncertain_marker_count,mio_count,mrd_count,negative_count,zero_count,min,median,max
0,Stamm-/Grundkapital EUR,share_capital,99603,98676,5423,98676,4496,926,1,0,0,81,2,0,0,1.000000e+00,25100.000,1.771093e+12
1,Bilanzsumme EUR,assets,98862,98861,5238,98861,5237,1,0,0,0,2392,0,0,0,1.300000e-01,1231556.470,2.913900e+12
2,Gewinn EUR,profit,84580,84579,19520,84579,19519,1,0,0,3847,149,0,15408,11675,-9.324879e+10,36210.220,1.103883e+10
3,Gewinn CAGR %,profit_cagr_pct,42981,42981,61118,42981,61118,0,0,0,1923,0,0,14513,64,-1.000000e+02,12.750,9.936200e+03
4,Umsatz EUR,revenue,98853,98852,5247,98852,5246,1,0,0,1,3086,4,3,149,-5.751900e+04,3300000.000,4.715487e+11
5,Umsatz CAGR %,revenue_cagr_pct,94990,94990,9109,94990,9109,0,0,0,4190,0,0,25874,4787,-1.000000e+02,5.200,9.909630e+03
6,Umsatzrendite %,profit_margin_original_pct,84363,84363,19736,84363,19736,0,0,0,3823,0,0,15266,12185,-7.754750e+03,1.000,9.801510e+03
7,Eigenkapital EUR,equity,98025,98024,6075,98024,6074,1,0,0,1631,1264,2,6365,1445,-4.017151e+11,395298.440,5.742590e+13
8,EK-Quote %,equity_ratio_original_pct,97982,97982,6117,97982,6117,0,0,0,1630,0,0,6348,1540,-8.021930e+03,43.685,9.474500e+03
9,EK-Rendite %,roe_original_pct,76660,76660,27439,76660,27439,0,0,0,2774,0,0,11825,10879,-9.872920e+03,8.895,9.936500e+03


**Cell 25 - Save numeric conversion audit**

In [29]:
conversion_audit_path = output_folder / "part2_numeric_conversion_audit.csv"

conversion_audit_df.to_csv(
    conversion_audit_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved numeric conversion audit to:")
print(conversion_audit_path)

Saved numeric conversion audit to:
/content/drive/MyDrive/IBS_7_Thesis/Python Files/cleaning_audit_outputs/part2_numeric_conversion_audit.csv


**Cell 26 - Show invalid or non-EUR examples**

In [30]:
for raw_col, clean_col in numeric_column_map.items():

    problem_mask = df_num[clean_col + "_parse_status"].isin([
        "non_eur_currency",
        "invalid_format",
        "conversion_error"
    ])

    problem_values = (
        df_num.loc[problem_mask, raw_col]
        .dropna()
        .astype(str)
        .drop_duplicates()
        .head(20)
        .tolist()
    )

    if len(problem_values) > 0:
        print("\n" + "="*80)
        print(f"Column: {raw_col}")
        print(f"Clean column: {clean_col}")
        print("Problem examples:")
        print(problem_values)


Column: Stamm-/Grundkapital EUR
Clean column: share_capital
Problem examples:
['50.000 DEM', '100.000 DEM', '18 Mio. DEM', '400.000 DEM', '1.000.000 DEM', '2.800.000 DEM', '500.000 DEM', '9.250.000 DEM', '54.000 DEM', '403 Mio. DEM', '1.865.000 DEM', '2.500.000 DEM', '1.500.000 DEM', '25,9 Mio. DEM', '10 Mio. DEM', '51.000 DEM', '1.200.000 DEM', '300.000 DEM', '85.000 DEM', '1.250.000 DEM']

Column: Bilanzsumme EUR
Clean column: assets
Problem examples:
['33,05 $']

Column: Gewinn EUR
Clean column: profit
Problem examples:
['7,77 $']

Column: Umsatz EUR
Clean column: revenue
Problem examples:
['257,33 $']

Column: Eigenkapital EUR
Clean column: equity
Problem examples:
['33,05 $']

Column: Umsatz pro Mitarbeiter EUR
Clean column: revenue_per_employee_original
Problem examples:
['2,80 $']

Column: Steuern EUR
Clean column: taxes
Problem examples:
['1,73 $']

Column: Kassenbestand EUR
Clean column: cash
Problem examples:
['54,57 $']

Column: Forderungen EUR
Clean column: receivables
Pro

**Cell 27 — Descriptive statistics of cleaned numeric columns**

In [31]:
clean_numeric_cols = list(numeric_column_map.values())

clean_numeric_summary = df_num[clean_numeric_cols].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
).T

clean_numeric_summary

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
share_capital,98676.0,2.228907e+07,5.704005e+09,1.000000e+00,2.000000e+00,1000.0000,25000.000,25100.000,5.030000e+04,6.372855e+05,6.052065e+06,1.771093e+12
assets,98861.0,1.196656e+08,1.097818e+10,1.300000e-01,1.032530e+04,52460.8700,489968.610,1231556.470,4.523259e+06,2.826013e+07,3.675855e+08,2.913900e+12
profit,84579.0,1.371382e+05,3.427518e+08,-9.324879e+10,-1.618252e+06,-185506.9350,0.000,36210.220,2.154636e+05,2.019834e+06,1.222238e+07,1.103883e+10
profit_cagr_pct,42981.0,8.604260e+01,4.525784e+02,-1.000000e+02,-9.089000e+01,-61.8400,-9.830,12.750,5.008000e+01,3.134700e+02,1.728188e+03,9.936200e+03
revenue,98852.0,8.413438e+07,2.769594e+09,-5.751900e+04,1.350593e+04,110000.0000,1200000.000,3300000.000,9.400000e+06,4.800000e+07,4.162184e+08,4.715487e+11
revenue_cagr_pct,94990.0,3.661237e+01,2.828597e+02,-1.000000e+02,-5.688330e+01,-21.3300,-1.340,5.200,1.691000e+01,1.015330e+02,6.272876e+02,9.909630e+03
profit_margin_original_pct,84363.0,1.372541e+01,2.224129e+02,-7.754750e+03,-2.587760e+01,-5.6800,0.000,1.000,4.500000e+00,3.885000e+01,1.366500e+02,9.801510e+03
equity,98024.0,5.991846e+08,1.834244e+11,-4.017151e+11,-1.138355e+06,-32081.6530,79071.520,395298.440,1.398496e+06,1.222535e+07,1.021778e+08,5.742590e+13
equity_ratio_original_pct,97982.0,4.320279e+01,8.792363e+01,-8.021930e+03,-7.148760e+01,-7.5395,13.560,43.685,7.337000e+01,9.726000e+01,1.000000e+02,9.474500e+03
roe_original_pct,76660.0,1.804008e+01,3.230369e+02,-9.872920e+03,-2.859789e+02,-36.5005,0.000,8.895,2.695000e+01,8.940100e+01,3.101000e+02,9.936500e+03


In [32]:
part2_output_path = output_folder / "DE_master_file_part2_numeric_converted.csv"

df_num.to_csv(
    part2_output_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved Part 2 numeric converted dataset to:")
print(part2_output_path)

Saved Part 2 numeric converted dataset to:
/content/drive/MyDrive/IBS_7_Thesis/Python Files/cleaning_audit_outputs/DE_master_file_part2_numeric_converted.csv


**Cell 29 — Quick check of key converted variables**

In [33]:
key_clean_cols = [
    "assets",
    "profit",
    "revenue",
    "equity",
    "liabilities",
    "cash",
    "receivables",
    "employees"
]

df_num[key_clean_cols].head(20)

,assets,profit,revenue,equity,liabilities,cash,receivables,employees
0,679793.84,-113496.26,690000.00,200063.46,358115.38,150845.95,59624.39,NaN
1,679593.17,-217337.80,4300000.00,619387.88,29415.29,333451.04,331284.83,NaN
2,679558.80,133323.10,1100000.00,156773.48,162438.70,425336.85,181877.25,12.0
3,679529.43,216008.68,1100000.00,501018.27,136972.16,619318.74,49954.88,NaN
4,679420.19,6529.39,3200000.00,534165.84,112454.35,162635.85,191789.74,13.0
5,679313.87,0.00,650000.00,202876.83,469937.04,99062.00,1282.57,NaN
6,679249.22,111731.97,2100000.00,484414.72,67902.77,334961.84,83703.89,NaN
7,679168.75,225663.59,2800000.00,380615.76,129109.51,455133.23,153923.52,NaN
8,679101.39,82210.60,1800000.00,385024.26,257234.13,386630.12,102142.26,12.0
9,679085.83,82402.34,579282.91,452985.03,12119.68,244002.65,130898.81,NaN


**Cell 30 — Check missing values after numeric conversion**

In [34]:
numeric_missing_after_conversion = pd.DataFrame({
    "clean_column": clean_numeric_cols,
    "non_missing": [df_num[col].notna().sum() for col in clean_numeric_cols],
    "missing": [df_num[col].isna().sum() for col in clean_numeric_cols],
    "missing_pct": [round(df_num[col].isna().mean() * 100, 2) for col in clean_numeric_cols]
})

numeric_missing_after_conversion.sort_values("missing_pct", ascending=False)

,clean_column,non_missing,missing,missing_pct
12,taxes,1325,102774,98.73
16,personnel_expense,8571,95528,91.77
10,employees,32754,71345,68.54
11,revenue_per_employee_original,32748,71351,68.54
3,profit_cagr_pct,42981,61118,58.71
9,roe_original_pct,76660,27439,26.36
6,profit_margin_original_pct,84363,19736,18.96
2,profit,84579,19520,18.75
13,cash,84772,19327,18.57
14,receivables,84967,19132,18.38


## **Part 3 - Creating clean thesis base and remove permanently unsusable rows**

**Cell 31 - Start part 3 from numeric-cleaned data**

In [35]:
df_part3 = df_num.copy()

print("Starting Part 3 shape:", df_part3.shape)

Starting Part 3 shape: (104099, 149)


**Cell 32 - Create helper function to log removed rows**

In [36]:
removed_rows_log = []

def remove_and_log(df, condition_to_remove, reason):
    """
    Removes rows from df based on condition_to_remove
    and stores removed rows with a reason.
    """
    global removed_rows_log

    rows_before = len(df)

    removed = df.loc[condition_to_remove].copy()
    kept = df.loc[~condition_to_remove].copy()

    rows_removed = len(removed)
    rows_after = len(kept)

    if rows_removed > 0:
        removed["removal_reason"] = reason
        removed_rows_log.append(removed)

    print(f"{reason}")
    print(f"Rows before:  {rows_before}")
    print(f"Rows removed: {rows_removed}")
    print(f"Rows after:   {rows_after}")
    print("-" * 60)

    return kept

**Cell 33 - Clean basic text columns**

In [37]:
text_cols_to_clean = [
    "Name",
    "Land",
    "Status",
    "Branche (WZ)",
    "Register-ID",
    "North Data URL",
    "Finanzkennzahlen Datum"
]

for col in text_cols_to_clean:
    if col in df_part3.columns:
        df_part3[col] = df_part3[col].astype("string").str.strip()

print("Text columns cleaned.")

Text columns cleaned.


**Cell 34 - Create financial date and financial year**

In [38]:
df_part3["financial_date"] = pd.to_datetime(
    df_part3["Finanzkennzahlen Datum"],
    format="%d.%m.%Y",
    errors="coerce"
)

df_part3["financial_year"] = df_part3["financial_date"].dt.year

print("Valid financial dates:", df_part3["financial_date"].notna().sum())
print("Missing/invalid financial dates:", df_part3["financial_date"].isna().sum())

df_part3["financial_year"].value_counts(dropna=False).sort_index()

Valid financial dates: 98912
Missing/invalid financial dates: 5187


,count
financial_year,
2006.0,5
2007.0,4
2008.0,4
2009.0,11
2010.0,27
2011.0,162
2012.0,95
2013.0,40
2014.0,53


**Cell 35 - Clean country and status variables**

In [39]:
df_part3["country_clean"] = df_part3["Land"].astype("string").str.upper().str.strip()

df_part3["status_clean"] = (
    df_part3["Status"]
    .astype("string")
    .str.lower()
    .str.strip()
)

# Harmonise common active-status wording
df_part3["status_group"] = np.select(
    [
        df_part3["status_clean"].isin(["aktiv", "active"]),
        df_part3["status_clean"].str.contains("liquidation|insolvenz|inactive|gelöscht|dissolved", na=False)
    ],
    [
        "active",
        "inactive_or_distress"
    ],
    default="other_or_unknown"
)

print("Country distribution:")
display(df_part3["country_clean"].value_counts(dropna=False).head(20).to_frame("count"))

print("Status group distribution:")
display(df_part3["status_group"].value_counts(dropna=False).to_frame("count"))

Country distribution:


,count
country_clean,
DE,104093
CH,2
AT,2
IT,1
BE,1


Status group distribution:


,count
status_group,
active,99879
other_or_unknown,2335
inactive_or_distress,1885


In [40]:
def extract_wz_code(value):
    if pd.isna(value):
        return pd.NA

    x = str(value).strip()

    # Extract first number-like WZ/NACE code, e.g. 46.90, 62.01, 10.71
    match = re.search(r"\d{1,2}(?:\.\d{1,2})?", x)

    if match:
        return match.group(0)

    return pd.NA


df_part3["industry_raw"] = df_part3["Branche (WZ)"].astype("string").str.strip()
df_part3["wz_code"] = df_part3["industry_raw"].apply(extract_wz_code)

print("Missing industry_raw:", df_part3["industry_raw"].isna().sum())
print("Missing wz_code:", df_part3["wz_code"].isna().sum())

display(df_part3[["industry_raw", "wz_code"]].head(20))

Missing industry_raw: 49
Missing wz_code: 49


,industry_raw,wz_code
0,71.12.9 Sonstige Ingenieurbüros,71.12
1,71.11.1 Architekturbüros für Hochbau,71.11
2,71.12.1 Ingenieurbüros für bautechnische Gesam...,71.12
3,71.12.1 Ingenieurbüros für bautechnische Gesam...,71.12
4,"71.20.0 Technische, physikalische und chemisch...",71.20
5,71.12.2 Ingenieurbüros für technische Fachplan...,71.12
6,71.12.2 Ingenieurbüros für technische Fachplan...,71.12
7,"71.20.0 Technische, physikalische und chemisch...",71.20
8,71.12.3 Vermessungsbüros,71.12
9,71.11.1 Architekturbüros für Hochbau,71.11


**Cell 37 - Create company identifier**

In [41]:
df_part3["company_key"] = pd.NA

if "North Data URL" in df_part3.columns:
    df_part3["company_key"] = df_part3["North Data URL"]

# If North Data URL is missing, use Register-ID
if "Register-ID" in df_part3.columns:
    df_part3["company_key"] = df_part3["company_key"].fillna(df_part3["Register-ID"])

# If still missing, use Name as fallback
if "Name" in df_part3.columns:
    df_part3["company_key"] = df_part3["company_key"].fillna(df_part3["Name"])

df_part3["company_key"] = df_part3["company_key"].astype("string").str.strip()

print("Missing company_key:", df_part3["company_key"].isna().sum())
print("Unique company_key:", df_part3["company_key"].nunique(dropna=True))

Missing company_key: 0
Unique company_key: 96227


### **Permanent row removal**

**Cell 38 - Remove exact duplicate rows**

In [42]:
duplicate_mask = df_part3.duplicated(keep="first")

df_part3 = remove_and_log(
    df_part3,
    duplicate_mask,
    "Exact duplicate row"
)

Exact duplicate row
Rows before:  104099
Rows removed: 6510
Rows after:   97589
------------------------------------------------------------


**Cell 39 - Keep only German companies**

In [43]:
non_de_mask = df_part3["country_clean"] != "DE"

df_part3 = remove_and_log(
    df_part3,
    non_de_mask,
    "Non-German company: country is not DE"
)

Non-German company: country is not DE
Rows before:  97589
Rows removed: 6
Rows after:   97583
------------------------------------------------------------


**Cell 40 - Remove rows without calid financial date**

In [44]:
invalid_date_mask = df_part3["financial_date"].isna()

df_part3 = remove_and_log(
    df_part3,
    invalid_date_mask,
    "Missing or invalid financial date"
)

Missing or invalid financial date
Rows before:  97583
Rows removed: 5183
Rows after:   92400
------------------------------------------------------------


**Cell 41 - Keep only financail years 2020-2024**

In [45]:
valid_years = [2020, 2021, 2022, 2023, 2024]

outside_year_mask = ~df_part3["financial_year"].isin(valid_years)

df_part3 = remove_and_log(
    df_part3,
    outside_year_mask,
    "Financial year outside 2020-2024"
)

Financial year outside 2020-2024
Rows before:  92400
Rows removed: 745
Rows after:   91655
------------------------------------------------------------


**Cell 42 - Remove rows without usable positive assets**

In [46]:
invalid_assets_mask = df_part3["assets"].isna() | (df_part3["assets"] <= 0)

df_part3 = remove_and_log(
    df_part3,
    invalid_assets_mask,
    "Missing or non-positive total assets"
)

Missing or non-positive total assets
Rows before:  91655
Rows removed: 38
Rows after:   91617
------------------------------------------------------------


**Cell 43 - Create removed rows log file**

In [47]:
if len(removed_rows_log) > 0:
    removed_rows_df = pd.concat(removed_rows_log, ignore_index=True)
else:
    removed_rows_df = pd.DataFrame()

removed_rows_path = output_folder / "part3_removed_rows_log.csv"

removed_rows_df.to_csv(
    removed_rows_path,
    index=False,
    encoding="utf-8-sig"
)

print("Removed rows log saved to:")
print(removed_rows_path)

print("Total removed rows logged:", len(removed_rows_df))

Removed rows log saved to:
/content/drive/MyDrive/IBS_7_Thesis/Python Files/cleaning_audit_outputs/part3_removed_rows_log.csv
Total removed rows logged: 12482


### **Creating clean thesis base columns**

**Cell 44 - Select useful columns for thesis base**

In [48]:
main_base_cols = [
    # Identifiers
    "company_key",
    "Name",
    "Register-ID",
    "North Data URL",

    # Country/status/year/industry
    "country_clean",
    "Status",
    "status_group",
    "financial_date",
    "financial_year",
    "industry_raw",
    "wz_code",

    # Clean financial variables
    "share_capital",
    "assets",
    "profit",
    "revenue",
    "equity",
    "liabilities",
    "cash",
    "receivables",
    "employees",

    # Original ratio variables for validation only
    "profit_margin_original_pct",
    "equity_ratio_original_pct",
    "roe_original_pct",
    "revenue_cagr_pct",
    "profit_cagr_pct"
]

# Keep only columns that exist
main_base_cols = [col for col in main_base_cols if col in df_part3.columns]

df_base = df_part3[main_base_cols].copy()

print("Base dataset shape:", df_base.shape)
df_base.head()

Base dataset shape: (91617, 25)


,company_key,Name,Register-ID,North Data URL,country_clean,Status,status_group,financial_date,financial_year,industry_raw,...,equity,liabilities,cash,receivables,employees,profit_margin_original_pct,equity_ratio_original_pct,roe_original_pct,revenue_cagr_pct,profit_cagr_pct
0,https://www.northdata.de/Knut%20R%C3%B6sch%20B...,Knut Rösch Baugrunduntersuchungen GmbH,HRB 7129 KI,https://www.northdata.de/Knut%20R%C3%B6sch%20B...,DE,aktiv,active,2024-12-31,2024.0,71.12.9 Sonstige Ingenieurbüros,...,200063.46,358115.38,150845.95,59624.39,NaN,-16.45,29.43,-56.73,-4.80,NaN
1,https://www.northdata.de/Soulworks%20Developme...,Soulworks Developments GmbH,HRB 288800,https://www.northdata.de/Soulworks%20Developme...,DE,aktiv,active,2022-12-31,2022.0,71.11.1 Architekturbüros für Hochbau,...,619387.88,29415.29,333451.04,331284.83,NaN,-5.05,91.14,-35.09,7.80,NaN
2,https://www.northdata.de/BEHR%20INGENIEURE%20G...,BEHR INGENIEURE GmbH,HRB 11586,https://www.northdata.de/BEHR%20INGENIEURE%20G...,DE,aktiv,active,2023-12-31,2023.0,71.12.1 Ingenieurbüros für bautechnische Gesam...,...,156773.48,162438.70,425336.85,181877.25,12.0,12.12,23.07,85.04,-4.09,0.62
3,https://www.northdata.de/GLASFAKTOR%20Ingenieu...,GLASFAKTOR Ingenieure GmbH,HRB 30711,https://www.northdata.de/GLASFAKTOR%20Ingenieu...,DE,aktiv,active,2023-11-30,2023.0,71.12.1 Ingenieurbüros für bautechnische Gesam...,...,501018.27,136972.16,619318.74,49954.88,NaN,19.64,73.73,43.11,4.85,12.29
4,https://www.northdata.de/Novum%20Analytik%20Gm...,Novum Analytik GmbH,HRB 723862,https://www.northdata.de/Novum%20Analytik%20Gm...,DE,aktiv,active,2023-12-31,2023.0,"71.20.0 Technische, physikalische und chemisch...",...,534165.84,112454.35,162635.85,191789.74,13.0,0.20,78.62,1.22,-2.22,NaN


**Cell 46 - Check year distribution after cleaning**

In [49]:
base_missing_summary = pd.DataFrame({
    "column": df_base.columns,
    "non_missing": [df_base[col].notna().sum() for col in df_base.columns],
    "missing": [df_base[col].isna().sum() for col in df_base.columns],
    "missing_pct": [round(df_base[col].isna().mean() * 100, 2) for col in df_base.columns]
})

base_missing_summary.sort_values("missing_pct", ascending=False)

,column,non_missing,missing,missing_pct
19,employees,30269,61348,66.96
24,profit_cagr_pct,39959,51658,56.38
22,roe_original_pct,71031,20586,22.47
20,profit_margin_original_pct,78010,13607,14.85
13,profit,78200,13417,14.64
17,cash,78462,13155,14.36
18,receivables,78611,13006,14.20
23,revenue_cagr_pct,88065,3552,3.88
16,liabilities,89553,2064,2.25
11,share_capital,89977,1640,1.79


**Cell 47 - Check status distribution after cleaning**

In [50]:
status_distribution = (
    df_base["status_group"]
    .value_counts(dropna=False)
    .reset_index()
)

status_distribution.columns = ["status_group", "rows"]

status_distribution

,status_group,rows
0,active,89952
1,inactive_or_distress,1624
2,other_or_unknown,41


**Cell 48 - Check industry coverage after cleaning**

In [51]:
industry_distribution = (
    df_base["wz_code"]
    .value_counts(dropna=False)
    .head(30)
    .reset_index()
)

industry_distribution.columns = ["wz_code", "rows"]

industry_distribution

,wz_code,rows
0,66.19,9571
1,71.12,8772
2,68.31,7778
3,66.22,3894
4,28.25,2935
5,71.11,2783
6,26.11,2589
7,45.19,2433
8,32.50,2403
9,45.11,2145


**Cell 49 - Save Part 3 base dataset**

In [52]:
part3_base_path = output_folder / "DE_analysis_base_part3_basic_cleaned.csv"

df_base.to_csv(
    part3_base_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved Part 3 base dataset to:")
print(part3_base_path)

print("Final Part 3 base shape:", df_base.shape)

Saved Part 3 base dataset to:
/content/drive/MyDrive/IBS_7_Thesis/Python Files/cleaning_audit_outputs/DE_analysis_base_part3_basic_cleaned.csv
Final Part 3 base shape: (91617, 25)


**Cell 50 - Create part 3 cleaning summary table**

In [53]:
part3_summary = []

part3_summary.append({
    "step": "Raw numeric-converted data",
    "rows": len(df_num)
})

part3_summary.append({
    "step": "After Part 3 permanent cleaning",
    "rows": len(df_base)
})

part3_summary_df = pd.DataFrame(part3_summary)

part3_summary_df

,step,rows
0,Raw numeric-converted data,104099
1,After Part 3 permanent cleaning,91617


In [54]:
df_base.shape


(91617, 25)

In [55]:
base_missing_summary.sort_values("missing_pct", ascending=False)

,column,non_missing,missing,missing_pct
19,employees,30269,61348,66.96
24,profit_cagr_pct,39959,51658,56.38
22,roe_original_pct,71031,20586,22.47
20,profit_margin_original_pct,78010,13607,14.85
13,profit,78200,13417,14.64
17,cash,78462,13155,14.36
18,receivables,78611,13006,14.20
23,revenue_cagr_pct,88065,3552,3.88
16,liabilities,89553,2064,2.25
11,share_capital,89977,1640,1.79


In [56]:
year_distribution = (
    df_base["financial_year"]
    .value_counts(dropna=False)
    .reset_index()
)

year_distribution.columns = ["financial_year", "rows"]

year_distribution

,financial_year,rows
0,2023.0,54910
1,2024.0,16138
2,2022.0,9109
3,2021.0,6718
4,2020.0,4742


In [57]:
status_distribution

,status_group,rows
0,active,89952
1,inactive_or_distress,1624
2,other_or_unknown,41


### **Part 4 - Apply financial validity rules**

Cell 51 - Start part 4

In [58]:
import pandas as pd
import numpy as np
from pathlib import Path

# Safety: make sure output folder exists
output_folder = Path("/content/drive/MyDrive/IBS_7_Thesis/Python Files/cleaning_audit_outputs")
output_folder.mkdir(parents=True, exist_ok=True)

# Start from Part 3 base
df_valid = df_base.copy()

print("Starting Part 4 shape:", df_valid.shape)

Starting Part 4 shape: (91617, 25)


We create a new dataset called df_valid.

df_base remains your basic cleaned dataset.

df_valid will become the financially checked dataset.


**Cell 52 - Create helper function for invalid values**

In [59]:
value_cleaning_log = []

def set_invalid_to_missing(df, col, invalid_mask, reason):
    """
    This function does NOT remove rows.
    It only sets impossible values in one column to missing.

    Example:
    If cash is negative, only cash becomes NaN.
    The company row remains available for other variables.
    """

    invalid_mask = invalid_mask.fillna(False)

    affected_count = int(invalid_mask.sum())

    examples = (
        df.loc[invalid_mask, col]
        .dropna()
        .head(10)
        .tolist()
    )

    value_cleaning_log.append({
        "column": col,
        "reason": reason,
        "affected_values": affected_count,
        "example_values": examples
    })

    print(f"{col}: {reason}")
    print(f"Affected values: {affected_count}")
    print(f"Example values: {examples}")
    print("-" * 70)

    df.loc[invalid_mask, col] = np.nan

    return df

**Cell 53 - Apply basic financial validity rules**

In [60]:
# 1. Revenue cannot be negative.
# Zero revenue can happen for holding companies or inactive companies,
# so we only remove negative revenue values, not zero revenue values.
df_valid = set_invalid_to_missing(
    df_valid,
    "revenue",
    df_valid["revenue"] < 0,
    "Negative revenue is not financially meaningful for ratio analysis"
)

# 2. Liabilities cannot be negative.
# A company can have zero liabilities, but negative liabilities are not meaningful here.
df_valid = set_invalid_to_missing(
    df_valid,
    "liabilities",
    df_valid["liabilities"] < 0,
    "Negative liabilities are not financially meaningful"
)

# 3. Cash cannot be negative.
df_valid = set_invalid_to_missing(
    df_valid,
    "cash",
    df_valid["cash"] < 0,
    "Negative cash balance is not financially meaningful"
)

# 4. Receivables cannot be negative.
df_valid = set_invalid_to_missing(
    df_valid,
    "receivables",
    df_valid["receivables"] < 0,
    "Negative receivables are not financially meaningful"
)

# 5. Employees must be at least 1 if used.
# Values below 1 are not useful for employee-based ratios.
df_valid = set_invalid_to_missing(
    df_valid,
    "employees",
    df_valid["employees"] < 1,
    "Employees below 1 are not useful for employee-based ratios"
)

# 6. Share capital should be positive if used.
# This is not a main thesis ratio variable, but we clean it for consistency.
df_valid = set_invalid_to_missing(
    df_valid,
    "share_capital",
    df_valid["share_capital"] <= 0,
    "Share capital must be positive if used"
)

revenue: Negative revenue is not financially meaningful for ratio analysis
Affected values: 3
Example values: [-9.0, -57519.0, -17040.0]
----------------------------------------------------------------------
liabilities: Negative liabilities are not financially meaningful
Affected values: 0
Example values: []
----------------------------------------------------------------------
cash: Negative cash balance is not financially meaningful
Affected values: 2
Example values: [-220967.5, -220967.5]
----------------------------------------------------------------------
receivables: Negative receivables are not financially meaningful
Affected values: 3
Example values: [-65221.0, -9868.68, -9868.68]
----------------------------------------------------------------------
employees: Employees below 1 are not useful for employee-based ratios
Affected values: 3
Example values: [0.75, 0.65, 0.5]
----------------------------------------------------------------------
share_capital: Share capital must b

**Cell 54 - Apply balance-sheet plausibility rules**

In [61]:
# 1. Cash cannot be larger than total assets.
# Cash is part of total assets, so cash > assets is not plausible.
df_valid = set_invalid_to_missing(
    df_valid,
    "cash",
    df_valid["cash"].notna() & df_valid["assets"].notna() & (df_valid["cash"] > df_valid["assets"]),
    "Cash is larger than total assets"
)

# 2. Receivables cannot be larger than total assets.
# Receivables are part of assets, so receivables > assets is not plausible.
df_valid = set_invalid_to_missing(
    df_valid,
    "receivables",
    df_valid["receivables"].notna() & df_valid["assets"].notna() & (df_valid["receivables"] > df_valid["assets"]),
    "Receivables are larger than total assets"
)

# 3. Positive equity cannot be larger than total assets if liabilities are non-negative.
# Since liabilities are cleaned to be non-negative, equity > assets is not plausible.
df_valid = set_invalid_to_missing(
    df_valid,
    "equity",
    df_valid["equity"].notna() & df_valid["assets"].notna() & (df_valid["equity"] > df_valid["assets"]),
    "Positive equity is larger than total assets"
)

cash: Cash is larger than total assets
Affected values: 56
Example values: [2051424150.0, 360504250.0, 56342.57, 129900000.0, 3513159.0, 9400.0, 1120.71, 779.06, 567124.0, 2653404790.0]
----------------------------------------------------------------------
receivables: Receivables are larger than total assets
Affected values: 51
Example values: [1258885.98, 130842.15, 3381360.0, 24871.24, 560316.43, 529356.0, 87252000.0, 9813591.57, 1534485.44, 133031.75]
----------------------------------------------------------------------
equity: Positive equity is larger than total assets
Affected values: 470
Example values: [1011262.68, 1047921.68, 1324923.14, 2861885000.0, 24685680.72, 58306537.67, 68434544.54, 35056000.0, 17000000.0, 24207343.94]
----------------------------------------------------------------------


**Cell 55 - Create warning flags for suspicious but not impossible cases**

In [62]:
# These flags are not used to delete data.
# They help us identify risky observations before ratio construction.

df_valid["flag_negative_equity"] = (
    df_valid["equity"].notna() &
    (df_valid["equity"] < 0)
)

df_valid["flag_liabilities_gt_assets"] = (
    df_valid["liabilities"].notna() &
    df_valid["assets"].notna() &
    (df_valid["liabilities"] > df_valid["assets"])
)

df_valid["flag_extreme_negative_equity_ratio"] = (
    df_valid["equity"].notna() &
    df_valid["assets"].notna() &
    ((df_valid["equity"] / df_valid["assets"]) < -5)
)

df_valid["flag_extreme_asset_turnover_precheck"] = (
    df_valid["revenue"].notna() &
    df_valid["assets"].notna() &
    (df_valid["revenue"] / df_valid["assets"] > 100)
)

df_valid["flag_extreme_liabilities_to_assets_precheck"] = (
    df_valid["liabilities"].notna() &
    df_valid["assets"].notna() &
    (df_valid["liabilities"] / df_valid["assets"] > 5)
)

warning_flags = [
    "flag_negative_equity",
    "flag_liabilities_gt_assets",
    "flag_extreme_negative_equity_ratio",
    "flag_extreme_asset_turnover_precheck",
    "flag_extreme_liabilities_to_assets_precheck"
]

warning_summary = pd.DataFrame({
    "flag": warning_flags,
    "count": [df_valid[col].sum() for col in warning_flags],
    "percentage": [round(df_valid[col].mean() * 100, 2) for col in warning_flags]
})

warning_summary

,flag,count,percentage
0,flag_negative_equity,5802,6.33
1,flag_liabilities_gt_assets,407,0.44
2,flag_extreme_negative_equity_ratio,36,0.04
3,flag_extreme_asset_turnover_precheck,177,0.19
4,flag_extreme_liabilities_to_assets_precheck,74,0.08


**Cell 56 - Check accounting consistency**

In [63]:
# Accounting identity approximation:
# Assets should roughly equal equity + liabilities.
# But scraped data may not contain all liability/equity components perfectly.
# Therefore, we only create a warning measure and do not remove rows.

df_valid["equity_plus_liabilities"] = df_valid["equity"] + df_valid["liabilities"]

df_valid["accounting_gap"] = (
    df_valid["assets"] - df_valid["equity_plus_liabilities"]
)

df_valid["accounting_gap_ratio"] = (
    df_valid["accounting_gap"].abs() / df_valid["assets"]
)

df_valid["flag_large_accounting_gap"] = (
    df_valid["accounting_gap_ratio"].notna() &
    (df_valid["accounting_gap_ratio"] > 1)
)

accounting_gap_summary = df_valid["accounting_gap_ratio"].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
)

accounting_gap_summary

,accounting_gap_ratio
count,88639.000000
mean,0.279919
std,9.265644
min,0.000000
1%,0.000000
5%,0.001608
25%,0.022750
50%,0.072243
75%,0.184507
95%,0.715268


**Cell 57 - Create component-quality summary after validity cleaning**

In [64]:
financial_component_cols = [
    "assets",
    "profit",
    "revenue",
    "equity",
    "liabilities",
    "cash",
    "receivables",
    "employees",
    "share_capital"
]

component_quality_summary = pd.DataFrame({
    "column": financial_component_cols,
    "non_missing": [df_valid[col].notna().sum() for col in financial_component_cols],
    "missing": [df_valid[col].isna().sum() for col in financial_component_cols],
    "missing_pct": [round(df_valid[col].isna().mean() * 100, 2) for col in financial_component_cols],
    "min": [df_valid[col].min(skipna=True) for col in financial_component_cols],
    "median": [df_valid[col].median(skipna=True) for col in financial_component_cols],
    "max": [df_valid[col].max(skipna=True) for col in financial_component_cols]
})

component_quality_summary

,column,non_missing,missing,missing_pct,min,median,max
0,assets,91617,0,0.00,1.000000e+00,1258780.32,2.913900e+12
1,profit,78200,13417,14.64,-9.324879e+10,36975.53,1.103883e+10
2,revenue,91590,27,0.03,0.000000e+00,3300000.00,4.715487e+11
3,equity,90343,1274,1.39,-4.017151e+11,393334.98,1.428010e+11
4,liabilities,89553,2064,2.25,0.000000e+00,371064.68,2.025742e+12
5,cash,78404,13213,14.42,0.000000e+00,182126.57,7.340912e+10
6,receivables,78557,13060,14.25,0.000000e+00,258271.09,6.860607e+10
7,employees,30266,61351,66.96,1.000000e+00,12.00,4.600000e+04
8,share_capital,89977,1640,1.79,1.000000e+00,25500.00,1.771093e+12


### **Part 4A - Diagnostic check before ratio consctruction**

**Cell 58 - Inspect extreme min/max rows**

In [65]:
# Part 4A starts here.
# Purpose:
# Before creating ratios, we inspect extreme values.
# This is important because very large values may be real,
# but they may also come from scraping errors, reporting scale issues, or special firms like banks.

df_check = df_valid.copy()

print("Dataset used for diagnostic check:", df_check.shape)

Dataset used for diagnostic check: (91617, 34)


**Cell 59 - Function to show extreme rows**

In [66]:
def show_extreme_rows(df, variable, n=10):
    """
    Shows the smallest and largest values for one financial variable.
    This helps us understand whether strange min/max values are real firms
    or likely data problems.
    """

    display_cols = [
        "Name",
        "financial_year",
        "status_group",
        "industry_raw",
        variable
    ]

    display_cols = [col for col in display_cols if col in df.columns]

    print("\n" + "="*90)
    print(f"Variable: {variable}")
    print("="*90)

    print("\nSmallest values:")
    display(
        df[df[variable].notna()]
        .sort_values(variable, ascending=True)
        [display_cols]
        .head(n)
    )

    print("\nLargest values:")
    display(
        df[df[variable].notna()]
        .sort_values(variable, ascending=False)
        [display_cols]
        .head(n)
    )

**Cell 60 - Check important financial extremes**

In [67]:
extreme_check_cols = [
    "assets",
    "profit",
    "revenue",
    "equity",
    "liabilities",
    "cash",
    "receivables",
    "employees",
    "share_capital"
]

for col in extreme_check_cols:
    show_extreme_rows(df_check, col, n=10)


Variable: assets

Smallest values:


,Name,financial_year,status_group,industry_raw,assets
14607,Zukunftsschmiede GmbH,2023.0,active,28.25.0 Herstellung von kälte- und lufttechnis...,1.00
6385,AnalyseBeratungService Finanzkonzept UG,2022.0,active,66.22.0 Tätigkeit von Versicherungsmaklerinnen...,1.00
101017,STERLING IMMOBILIEN UG,2023.0,active,"68.31.1 Vermittlung von Wohngrundstücken, Wohn...",1.00
6384,Jensten Brokers Europe GmbH,2024.0,active,66.22.0 Tätigkeit von Versicherungsmaklerinnen...,1.41
7303,Dotzauer Industrielackierung GmbH,2023.0,active,25.61.0 Oberflächenveredlung und Wärmebehandlung,1.83
6383,Adjurcon UG,2021.0,active,66.22.0 Tätigkeit von Versicherungsmaklerinnen...,2.00
14606,Hoffmann Consorten Hamburg GmbH,2024.0,active,28.25.0 Herstellung von kälte- und lufttechnis...,9.00
100136,Real Estate Services I.R. UG,2021.0,other_or_unknown,"68.31.1 Vermittlung von Wohngrundstücken, Wohn...",12.91
6382,GVF Gesellschaft für die Vermittlung von Finan...,2022.0,active,66.22.0 Tätigkeit von Versicherungsmaklerinnen...,18.20
59385,JSB UG,2024.0,inactive_or_distress,66.19.0 Sonstige mit Finanzdienstleistungen ve...,18.68



Largest values:


,Name,financial_year,status_group,industry_raw,assets
11696,Industrial and Commercial Bank of China Ltd. F...,2021.0,active,64.19.1 Kreditbanken einschließlich Zweigstell...,2.913900e+12
11738,Access Microfinance Holding AG,2023.0,active,64.99.9 Sonstige Finanzierungsinstitutionen a....,1.053013e+12
6386,Allianz SE,2024.0,active,66.22 Tätigkeit von Versicherungsmaklerinnen u...,1.044578e+12
11732,Hercegfi CNC Fertigungstechnik GmbH,2023.0,active,28.41.0 Herstellung von Werkzeugmaschinen für ...,9.307468e+11
12098,MÜNCHENSTIFT GmbH Gemeinnützige Gesellschaft d...,2024.0,active,88.10.1 Ambulante soziale Dienste,3.094541e+11
80110,EUROGATE Container Terminal Wilhelmshaven GmbH...,2023.0,active,52.24.0 Frachtumschlag,2.194019e+11
12001,RENA Technologies GmbH,2020.0,active,28.99.0 Herstellung von Maschinen für sonstige...,2.110846e+11
55255,ING Holding Deutschland GmbH,2024.0,active,64.20 Beteiligungsgesellschaften,2.004430e+11
7304,Beam Suntory Deutschland GmbH,2024.0,active,47.78 Sonstiger Einzelhandel in Verkaufsräumen...,1.145772e+11
11705,Robert Bosch GmbH,2023.0,active,26.30.0 Herstellung von Geräten und Einrichtun...,1.083300e+11



Variable: profit

Smallest values:


...

,Name,financial_year,status_group,industry_raw,employees
6472,Willis Towers Watson Versicherungsmakler GmbH,2023.0,active,66.22.0 Tätigkeit von Versicherungsmaklerinnen...,46000.0
15110,PHOENIX Pharma SE,2024.0,active,21.20.0 Herstellung von pharmazeutischen Spezi...,41276.0
7306,Fielmann Group AG,2024.0,active,47.78.1 Augenoptiker,24363.0
28354,Ford-Werke GmbH,2023.0,active,"71.20 Technische, physikalische und chemische ...",18405.0
11822,Diehl Stiftung & Co. KG,2024.0,active,30.30.0 Luft- und Raumfahrzeugbau,18379.0
11823,Diehl Verwaltungs-Stiftung HRA Juristische Person,2024.0,active,"84.13.0 Wirtschaftsförderung, -ordnung und -au...",18379.0
6396,R+V Allgemeine Versicherung AG,2023.0,active,66.22 Tätigkeit von Versicherungsmaklerinnen u...,11533.0
6395,ERGO Group AG,2023.0,active,66.22.0 Tätigkeit von Versicherungsmaklerinnen...,9022.0
28365,Zhongding Europe GmbH,2022.0,active,45.31.0 Großhandel mit Kraftwagenteilen und -z...,8662.0
17690,Goldschmitt techmobil GmbH,2024.0,active,45.19.0 Handel mit Kraftwagen mit einem Gesamt...,8400.0



Variable: share_capital

Smallest values:


,Name,financial_year,status_group,industry_raw,share_capital
102858,FPB Immobilien-Management UG,2023.0,active,"68.31.1 Vermittlung von Wohngrundstücken, Wohn...",1.0
103594,PA Immoconsult UG,2022.0,active,"68.31.1 Vermittlung von Wohngrundstücken, Wohn...",1.0
102889,planbar Kassel UG,2020.0,other_or_unknown,"68.31.1 Vermittlung von Wohngrundstücken, Wohn...",1.0
103354,HABRICH Immobilien UG,2022.0,active,"68.31.1 Vermittlung von Wohngrundstücken, Wohn...",1.0
101486,Vario-Trade UG,2021.0,active,"68.31.1 Vermittlung von Wohngrundstücken, Wohn...",1.0
101506,liberté Handels- und Vertriebsgesellschaft UG,2024.0,active,"68.31.1 Vermittlung von Wohngrundstücken, Wohn...",1.0
100670,INSTYLE Immobilien UG,2022.0,active,"68.31.1 Vermittlung von Wohngrundstücken, Wohn...",1.0
101558,VITA PG Hultschiner UG,2020.0,other_or_unknown,"68.31.1 Vermittlung von Wohngrundstücken, Wohn...",1.0
100673,Beilke Verwaltungs UG,2023.0,active,"68.31.1 Vermittlung von Wohngrundstücken, Wohn...",1.0
100681,PG BM 86 UG,2022.0,active,"68.31.1 Vermittlung von Wohngrundstücken, Wohn...",1.0



Largest values:


,Name,financial_year,status_group,industry_raw,share_capital
11756,Sumitomo Mitsui Banking Corporation Fil. Düsse...,2024.0,active,64.19.1 Kreditbanken einschließlich Zweigstell...,1.771093e+12
11696,Industrial and Commercial Bank of China Ltd. F...,2021.0,active,64.19.1 Kreditbanken einschließlich Zweigstell...,2.480000e+11
31438,First Commercial Bank Ltd. Frankfurt Branch,2024.0,active,64.19.1 Kreditbanken einschließlich Zweigstell...,1.104010e+11
11697,Morgan Stanley Europe SE,2024.0,active,64.99 Erbringung von sonstigen Finanzdienstlei...,3.901000e+09
18729,Norddeutsche Landesbank - Girozentrale - HRA J...,2023.0,active,66.19 Sonstige mit Finanzdienstleistungen verb...,3.170000e+09
11700,Hoechst GmbH,2023.0,active,64.20.0 Beteiligungsgesellschaften,1.429454e+09
11705,Robert Bosch GmbH,2023.0,active,26.30.0 Herstellung von Geräten und Einrichtun...,1.200000e+09
6386,Allianz SE,2024.0,active,66.22 Tätigkeit von Versicherungsmaklerinnen u...,1.169920e+09
12095,Uniper Global Commodities SE,2024.0,active,70.10 Verwaltung und Führung von Unternehmen u...,1.138188e+09
80114,KfW IPEX-Bank GmbH,2024.0,active,64.19.1 Kreditbanken einschließlich Zweigstell...,1.100000e+09


### **Part 4B - Improve industry classification**

**Cell 61 - Create better WZ/NACE industry variables**

In [68]:
def extract_wz_detailed(value):
    """
    Extracts the leading WZ code from the raw industry string.

    Example:
    '68.31.1 Vermittlung von Wohngrundstücken...' -> '68.31.1'
    '66.19 Sonstige mit Finanzdienstleistungen...' -> '66.19'
    """

    if pd.isna(value):
        return pd.NA

    x = str(value).strip()

    match = re.match(r"^(\d{1,2}(?:\.\d{1,2}){0,2})\b", x)

    if match:
        return match.group(1)

    return pd.NA


def create_wz_4digit(code):
    """
    Converts detailed WZ code to 4-digit class level.

    Example:
    68.31.1 -> 68.31
    66.19.0 -> 66.19
    """

    if pd.isna(code):
        return pd.NA

    parts = str(code).split(".")

    if len(parts) >= 2:
        return parts[0].zfill(2) + "." + parts[1].zfill(2)

    return parts[0].zfill(2)


def create_wz_2digit(code):
    """
    Converts WZ code to broad 2-digit industry division.

    Example:
    68.31.1 -> 68
    66.19.0 -> 66
    """

    if pd.isna(code):
        return pd.NA

    return str(code).split(".")[0].zfill(2)


def remove_wz_code_from_label(value):
    """
    Keeps only the industry label text.

    Example:
    '68.31.1 Vermittlung von...' -> 'Vermittlung von...'
    """

    if pd.isna(value):
        return pd.NA

    x = str(value).strip()
    label = re.sub(r"^\d{1,2}(?:\.\d{1,2}){0,2}\s*", "", x)

    return label.strip()


df_check["wz_code_detailed"] = df_check["industry_raw"].apply(extract_wz_detailed)
df_check["wz_4digit"] = df_check["wz_code_detailed"].apply(create_wz_4digit)
df_check["wz_2digit"] = df_check["wz_code_detailed"].apply(create_wz_2digit)
df_check["industry_label"] = df_check["industry_raw"].apply(remove_wz_code_from_label)

df_check[["industry_raw", "wz_code_detailed", "wz_4digit", "wz_2digit", "industry_label"]].head(20)

,industry_raw,wz_code_detailed,wz_4digit,wz_2digit,industry_label
0,71.12.9 Sonstige Ingenieurbüros,71.12.9,71.12,71,Sonstige Ingenieurbüros
1,71.11.1 Architekturbüros für Hochbau,71.11.1,71.11,71,Architekturbüros für Hochbau
2,71.12.1 Ingenieurbüros für bautechnische Gesam...,71.12.1,71.12,71,Ingenieurbüros für bautechnische Gesamtplanung
3,71.12.1 Ingenieurbüros für bautechnische Gesam...,71.12.1,71.12,71,Ingenieurbüros für bautechnische Gesamtplanung
4,"71.20.0 Technische, physikalische und chemisch...",71.20.0,71.20,71,"Technische, physikalische und chemische Unters..."
5,71.12.2 Ingenieurbüros für technische Fachplan...,71.12.2,71.12,71,Ingenieurbüros für technische Fachplanung und ...
6,71.12.2 Ingenieurbüros für technische Fachplan...,71.12.2,71.12,71,Ingenieurbüros für technische Fachplanung und ...
7,"71.20.0 Technische, physikalische und chemisch...",71.20.0,71.20,71,"Technische, physikalische und chemische Unters..."
8,71.12.3 Vermessungsbüros,71.12.3,71.12,71,Vermessungsbüros
9,71.11.1 Architekturbüros für Hochbau,71.11.1,71.11,71,Architekturbüros für Hochbau


**Cell 62 - Create WZ sector groups**

In [69]:
def assign_wz_section(wz_2digit):
    """
    Assigns broad WZ/NACE section based on 2-digit code.
    This is useful for high-level descriptive tables and robustness checks.
    """

    if pd.isna(wz_2digit):
        return pd.NA

    try:
        code = int(wz_2digit)
    except:
        return pd.NA

    if 1 <= code <= 3:
        return "A Agriculture, forestry and fishing"
    elif 5 <= code <= 9:
        return "B Mining and quarrying"
    elif 10 <= code <= 33:
        return "C Manufacturing"
    elif code == 35:
        return "D Energy supply"
    elif 36 <= code <= 39:
        return "E Water, sewerage and waste"
    elif 41 <= code <= 43:
        return "F Construction"
    elif 45 <= code <= 47:
        return "G Wholesale and retail trade"
    elif 49 <= code <= 53:
        return "H Transport and storage"
    elif 55 <= code <= 56:
        return "I Accommodation and food service"
    elif 58 <= code <= 63:
        return "J Information and communication"
    elif 64 <= code <= 66:
        return "K Financial and insurance activities"
    elif code == 68:
        return "L Real estate activities"
    elif 69 <= code <= 75:
        return "M Professional, scientific and technical activities"
    elif 77 <= code <= 82:
        return "N Administrative and support service activities"
    elif code == 84:
        return "O Public administration"
    elif code == 85:
        return "P Education"
    elif 86 <= code <= 88:
        return "Q Human health and social work"
    elif 90 <= code <= 93:
        return "R Arts, entertainment and recreation"
    elif 94 <= code <= 96:
        return "S Other service activities"
    else:
        return "Other / unknown"


df_check["wz_section"] = df_check["wz_2digit"].apply(assign_wz_section)

# Special flags for later analysis
df_check["is_financial_sector"] = df_check["wz_2digit"].isin(["64", "65", "66"])
df_check["is_real_estate_sector"] = df_check["wz_2digit"].isin(["68"])
df_check["is_manufacturing_sector"] = df_check["wz_2digit"].between("10", "33")

df_check[["wz_2digit", "wz_section", "is_financial_sector"]].head()

,wz_2digit,wz_section,is_financial_sector
0,71,"M Professional, scientific and technical activ...",False
1,71,"M Professional, scientific and technical activ...",False
2,71,"M Professional, scientific and technical activ...",False
3,71,"M Professional, scientific and technical activ...",False
4,71,"M Professional, scientific and technical activ...",False


**Cell 63 - Check industry distribution at different levels**

In [70]:
print("Top WZ detailed codes:")
display(
    df_check["wz_code_detailed"]
    .value_counts(dropna=False)
    .head(30)
    .reset_index()
    .rename(columns={"index": "wz_code_detailed", "count": "rows"})
)

print("Top WZ 4-digit codes:")
display(
    df_check["wz_4digit"]
    .value_counts(dropna=False)
    .head(30)
    .reset_index()
    .rename(columns={"index": "wz_4digit", "count": "rows"})
)

print("Top WZ 2-digit divisions:")
display(
    df_check["wz_2digit"]
    .value_counts(dropna=False)
    .head(30)
    .reset_index()
    .rename(columns={"index": "wz_2digit", "count": "rows"})
)

print("Broad WZ section distribution:")
display(
    df_check["wz_section"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "wz_section", "count": "rows"})
)

Top WZ detailed codes:


,wz_code_detailed,rows
0,66.19.0,9559
1,68.31.1,7755
2,66.22.0,3849
3,71.12.1,3268
4,71.12.2,3212
5,28.25.0,2935
6,26.11.9,2540
7,45.11.0,2141
8,28.99.0,2006
9,25.62.0,1864


Top WZ 4-digit codes:


,wz_4digit,rows
0,66.19,9571
1,71.12,8772
2,68.31,7778
3,66.22,3894
4,28.25,2935
5,71.11,2783
6,26.11,2589
7,45.19,2433
8,32.50,2403
9,45.11,2145


Top WZ 2-digit divisions:


,wz_2digit,rows
0,66,13540
1,71,12996
2,28,12469
3,25,10323
4,68,8918
5,45,7984
6,26,6300
7,47,3000
8,77,2920
9,32,2415


Broad WZ section distribution:


,wz_section,rows
0,C Manufacturing,35810
1,K Financial and insurance activities,14973
2,"M Professional, scientific and technical activ...",13627
3,G Wholesale and retail trade,11796
4,L Real estate activities,8918
5,N Administrative and support service activities,3068
6,F Construction,1003
7,D Energy supply,744
8,"E Water, sewerage and waste",677
9,J Information and communication,284


**Cell 64 - Create readable industry table with labels**

In [72]:
# For each 4-digit WZ code, choose the most common industry label.
# This makes your thesis tables easier to understand.

industry_label_map = (
    df_check
    .dropna(subset=["wz_4digit", "industry_label"])
    .groupby(["wz_4digit", "industry_label"])
    .size()
    .reset_index(name="label_count")
    .sort_values(["wz_4digit", "label_count"], ascending=[True, False])
    .drop_duplicates(subset=["wz_4digit"])
    [["wz_4digit", "industry_label"]]
)

industry_distribution_4digit = (
    df_check
    .groupby("wz_4digit", dropna=False)
    .size()
    .reset_index(name="rows")
    .sort_values("rows", ascending=False)
    .merge(industry_label_map, on="wz_4digit", how="left")
)

industry_distribution_4digit.head(30)

,wz_4digit,rows,industry_label
0,66.19,9571,Sonstige mit Finanzdienstleistungen verbundene...
1,71.12,8772,Ingenieurbüros für bautechnische Gesamtplanung
2,68.31,7778,"Vermittlung von Wohngrundstücken, Wohngebäuden..."
3,66.22,3894,Tätigkeit von Versicherungsmaklerinnen und -ma...
4,28.25,2935,Herstellung von kälte- und lufttechnischen Erz...
5,71.11,2783,Architekturbüros für Hochbau
6,26.11,2589,Herstellung von sonstigen elektronischen Bauel...
7,45.19,2433,Handel mit Kraftwagen mit einem Gesamtgewicht ...
8,32.50,2403,Zahntechnische Laboratorien
9,45.11,2145,Handel mit Kraftwagen mit einem Gesamtgewicht ...


### **Part 4C - Add denominator-quality rules**

**Cell 65 - Add denominator qauality flags**

In [71]:
# Small denominators create unstable ratios.
# Example:
# profit / revenue becomes meaningless if revenue is 1 EUR.
#
# We do not delete the row here.
# We only create flags that will be used during ratio construction.

MIN_EUR_DENOMINATOR = 1000
MIN_EMPLOYEES = 1

df_check["assets_denominator_ok"] = (
    df_check["assets"].notna() &
    (df_check["assets"] >= MIN_EUR_DENOMINATOR)
)

df_check["revenue_denominator_ok"] = (
    df_check["revenue"].notna() &
    (df_check["revenue"] >= MIN_EUR_DENOMINATOR)
)

df_check["equity_denominator_ok"] = (
    df_check["equity"].notna() &
    (df_check["equity"] >= MIN_EUR_DENOMINATOR)
)

df_check["liabilities_denominator_ok"] = (
    df_check["liabilities"].notna() &
    (df_check["liabilities"] >= MIN_EUR_DENOMINATOR)
)

df_check["receivables_denominator_ok"] = (
    df_check["receivables"].notna() &
    (df_check["receivables"] >= MIN_EUR_DENOMINATOR)
)

df_check["employees_denominator_ok"] = (
    df_check["employees"].notna() &
    (df_check["employees"] >= MIN_EMPLOYEES)
)

denominator_quality_summary = pd.DataFrame({
    "denominator": [
        "assets",
        "revenue",
        "equity",
        "liabilities",
        "receivables",
        "employees"
    ],
    "minimum_required": [
        MIN_EUR_DENOMINATOR,
        MIN_EUR_DENOMINATOR,
        MIN_EUR_DENOMINATOR,
        MIN_EUR_DENOMINATOR,
        MIN_EUR_DENOMINATOR,
        MIN_EMPLOYEES
    ],
    "usable_rows": [
        df_check["assets_denominator_ok"].sum(),
        df_check["revenue_denominator_ok"].sum(),
        df_check["equity_denominator_ok"].sum(),
        df_check["liabilities_denominator_ok"].sum(),
        df_check["receivables_denominator_ok"].sum(),
        df_check["employees_denominator_ok"].sum()
    ],
    "not_usable_rows": [
        len(df_check) - df_check["assets_denominator_ok"].sum(),
        len(df_check) - df_check["revenue_denominator_ok"].sum(),
        len(df_check) - df_check["equity_denominator_ok"].sum(),
        len(df_check) - df_check["liabilities_denominator_ok"].sum(),
        len(df_check) - df_check["receivables_denominator_ok"].sum(),
        len(df_check) - df_check["employees_denominator_ok"].sum()
    ],
    "usable_pct": [
        round(df_check["assets_denominator_ok"].mean() * 100, 2),
        round(df_check["revenue_denominator_ok"].mean() * 100, 2),
        round(df_check["equity_denominator_ok"].mean() * 100, 2),
        round(df_check["liabilities_denominator_ok"].mean() * 100, 2),
        round(df_check["receivables_denominator_ok"].mean() * 100, 2),
        round(df_check["employees_denominator_ok"].mean() * 100, 2)
    ]
})

denominator_quality_summary

,denominator,minimum_required,usable_rows,not_usable_rows,usable_pct
0,assets,1000,91503,114,99.88
1,revenue,1000,91358,259,99.72
2,equity,1000,82685,8932,90.25
3,liabilities,1000,86890,4727,94.84
4,receivables,1000,76267,15350,83.25
5,employees,1,30266,61351,33.04


**Cell 66 - Update ratio feasibility using denominator rules**

In [73]:
ratio_feasibility_rules_v2 = {
    "profit_margin": (
        df_check["profit"].notna() &
        df_check["revenue_denominator_ok"]
    ),

    "roa": (
        df_check["profit"].notna() &
        df_check["assets_denominator_ok"]
    ),

    "roe": (
        df_check["profit"].notna() &
        df_check["equity_denominator_ok"]
    ),

    "equity_ratio_calc": (
        df_check["equity"].notna() &
        df_check["assets_denominator_ok"]
    ),

    "debt_ratio": (
        df_check["liabilities"].notna() &
        df_check["assets_denominator_ok"]
    ),

    "cash_to_assets": (
        df_check["cash"].notna() &
        df_check["assets_denominator_ok"]
    ),

    "receivables_to_assets": (
        df_check["receivables"].notna() &
        df_check["assets_denominator_ok"]
    ),

    "revenue_per_employee_calc": (
        df_check["revenue"].notna() &
        df_check["employees_denominator_ok"]
    ),

    "profit_per_employee": (
        df_check["profit"].notna() &
        df_check["employees_denominator_ok"]
    ),

    "asset_turnover": (
        df_check["revenue"].notna() &
        df_check["assets_denominator_ok"]
    ),

    "debt_to_equity_ratio": (
        df_check["liabilities"].notna() &
        df_check["equity_denominator_ok"]
    ),

    "cash_ratio": (
        df_check["cash"].notna() &
        df_check["liabilities_denominator_ok"]
    ),

    "receivables_turnover": (
        df_check["revenue"].notna() &
        df_check["receivables_denominator_ok"]
    )
}

ratio_feasibility_df_v2 = pd.DataFrame({
    "ratio": list(ratio_feasibility_rules_v2.keys()),
    "usable_rows": [mask.sum() for mask in ratio_feasibility_rules_v2.values()],
    "not_usable_rows": [len(df_check) - mask.sum() for mask in ratio_feasibility_rules_v2.values()],
    "usable_pct": [round(mask.mean() * 100, 2) for mask in ratio_feasibility_rules_v2.values()]
})

ratio_feasibility_df_v2.sort_values("usable_rows", ascending=False)

,ratio,usable_rows,not_usable_rows,usable_pct
9,asset_turnover,91476,141,99.85
3,equity_ratio_calc,90247,1370,98.50
4,debt_ratio,89484,2133,97.67
10,debt_to_equity_ratio,81074,10543,88.49
6,receivables_to_assets,78529,13088,85.71
5,cash_to_assets,78359,13258,85.53
1,roa,78139,13478,85.29
0,profit_margin,77983,13634,85.12
12,receivables_turnover,76249,15368,83.23
11,cash_ratio,75593,16024,82.51


**Cell 67 - Save improved Part 4A dataset**

In [74]:
part4a_checked_path = output_folder / "DE_analysis_base_part4a_checked_with_industry_and_denominator_flags.csv"
industry_distribution_path = output_folder / "part4a_industry_distribution_4digit.csv"
denominator_quality_path = output_folder / "part4a_denominator_quality_summary.csv"
ratio_feasibility_v2_path = output_folder / "part4a_ratio_feasibility_with_denominator_rules.csv"

df_check.to_csv(part4a_checked_path, index=False, encoding="utf-8-sig")
industry_distribution_4digit.to_csv(industry_distribution_path, index=False, encoding="utf-8-sig")
denominator_quality_summary.to_csv(denominator_quality_path, index=False, encoding="utf-8-sig")
ratio_feasibility_df_v2.to_csv(ratio_feasibility_v2_path, index=False, encoding="utf-8-sig")

print("Saved checked dataset:")
print(part4a_checked_path)

print("\nSaved industry distribution:")
print(industry_distribution_path)

print("\nSaved denominator quality summary:")
print(denominator_quality_path)

print("\nSaved updated ratio feasibility table:")
print(ratio_feasibility_v2_path)

print("\nFinal checked dataset shape:", df_check.shape)

Saved checked dataset:
/content/drive/MyDrive/IBS_7_Thesis/Python Files/cleaning_audit_outputs/DE_analysis_base_part4a_checked_with_industry_and_denominator_flags.csv

Saved industry distribution:
/content/drive/MyDrive/IBS_7_Thesis/Python Files/cleaning_audit_outputs/part4a_industry_distribution_4digit.csv

Saved denominator quality summary:
/content/drive/MyDrive/IBS_7_Thesis/Python Files/cleaning_audit_outputs/part4a_denominator_quality_summary.csv

Saved updated ratio feasibility table:
/content/drive/MyDrive/IBS_7_Thesis/Python Files/cleaning_audit_outputs/part4a_ratio_feasibility_with_denominator_rules.csv

Final checked dataset shape: (91617, 48)


**Extreme values should not be manually deleted now.** Some large values are real financial/holding companies, but some may be suspicious. The best thesis-safe approach is:

keep original cleaned values,

create ratios only when denominator is reliable,

apply hard financial plausibility limits to ratios,

winsorize final ratios at 1% and 99%,

later run a robustness check excluding financial-sector firms.

### **Part 5 - Construct ratio variables safely**

**Cell 68 - Start ratio construction dataset**

In [79]:
# ============================================================
# PART 5: SAFE RATIO CONSTRUCTION
# ============================================================
# Goal:
# Create all ratio variables needed for thesis analysis.
#
# Important principle:
# We do not overwrite original financial variables.
# We create ratio variables separately and only calculate them
# when the denominator is meaningful.
# ============================================================

# Use df_check if it exists because it already contains improved industry variables.
# Otherwise, use df_valid from Part 4.
if "df_check" in globals():
    df_ratio = df_check.copy()
else:
    df_ratio = df_valid.copy()

print("Starting Part 5 dataset shape:", df_ratio.shape)

Starting Part 5 dataset shape: (91617, 48)


**Cell 69 - Define safe division function **

In [81]:
def safe_divide(numerator, denominator, valid_mask):
    """
    Calculates numerator / denominator only when valid_mask is True.
    Otherwise returns NaN.

    This prevents invalid ratios from being created when:
    - denominator is missing,
    - denominator is zero,
    - denominator is too small,
    - variable is not meaningful for that ratio.
    """

    result = pd.Series(np.nan, index=numerator.index, dtype="float64")
    result.loc[valid_mask] = numerator.loc[valid_mask] / denominator.loc[valid_mask]
    return result

**Cell 70 - Construct raw ratio vaiables**

In [82]:
# ============================================================
# Ratio construction
# ============================================================
# These are raw constructed ratios.
# They are not winsorized yet.
# We will clean extreme ratio values in the next step.
# ============================================================

# 1. Profit margin = profit / revenue
df_ratio["profit_margin"] = safe_divide(
    df_ratio["profit"],
    df_ratio["revenue"],
    df_ratio["profit"].notna() & df_ratio["revenue_denominator_ok"]
)

# 2. Return on assets = profit / assets
df_ratio["roa"] = safe_divide(
    df_ratio["profit"],
    df_ratio["assets"],
    df_ratio["profit"].notna() & df_ratio["assets_denominator_ok"]
)

# 3. Return on equity = profit / equity
# We use only positive and meaningful equity.
# Negative equity firms are real, but ROE becomes hard to interpret for them.
df_ratio["roe"] = safe_divide(
    df_ratio["profit"],
    df_ratio["equity"],
    df_ratio["profit"].notna() & df_ratio["equity_denominator_ok"]
)

# 4. Equity ratio = equity / assets
df_ratio["equity_ratio_calc"] = safe_divide(
    df_ratio["equity"],
    df_ratio["assets"],
    df_ratio["equity"].notna() & df_ratio["assets_denominator_ok"]
)

# 5. Debt ratio = liabilities / assets
df_ratio["debt_ratio"] = safe_divide(
    df_ratio["liabilities"],
    df_ratio["assets"],
    df_ratio["liabilities"].notna() & df_ratio["assets_denominator_ok"]
)

# 6. Cash to assets = cash / assets
df_ratio["cash_to_assets"] = safe_divide(
    df_ratio["cash"],
    df_ratio["assets"],
    df_ratio["cash"].notna() & df_ratio["assets_denominator_ok"]
)

# 7. Receivables to assets = receivables / assets
df_ratio["receivables_to_assets"] = safe_divide(
    df_ratio["receivables"],
    df_ratio["assets"],
    df_ratio["receivables"].notna() & df_ratio["assets_denominator_ok"]
)

# 8. Revenue per employee = revenue / employees
df_ratio["revenue_per_employee_calc"] = safe_divide(
    df_ratio["revenue"],
    df_ratio["employees"],
    df_ratio["revenue"].notna() & df_ratio["employees_denominator_ok"]
)

# 9. Profit per employee = profit / employees
df_ratio["profit_per_employee"] = safe_divide(
    df_ratio["profit"],
    df_ratio["employees"],
    df_ratio["profit"].notna() & df_ratio["employees_denominator_ok"]
)

# 10. Asset turnover = revenue / assets
df_ratio["asset_turnover"] = safe_divide(
    df_ratio["revenue"],
    df_ratio["assets"],
    df_ratio["revenue"].notna() & df_ratio["assets_denominator_ok"]
)

# 11. Debt to equity = liabilities / equity
df_ratio["debt_to_equity_ratio"] = safe_divide(
    df_ratio["liabilities"],
    df_ratio["equity"],
    df_ratio["liabilities"].notna() & df_ratio["equity_denominator_ok"]
)

# 12. Cash ratio proxy = cash / liabilities
# Important: this is not the classical accounting cash ratio,
# because we do not have current liabilities.
# In the thesis, call it "cash-to-liabilities proxy".
df_ratio["cash_ratio"] = safe_divide(
    df_ratio["cash"],
    df_ratio["liabilities"],
    df_ratio["cash"].notna() & df_ratio["liabilities_denominator_ok"]
)

# Extra clear name for thesis interpretation
df_ratio["cash_to_liabilities"] = df_ratio["cash_ratio"]

# 13. Receivables turnover proxy = revenue / receivables
# Normally this should use average receivables, but we only have one-year values.
df_ratio["receivables_turnover"] = safe_divide(
    df_ratio["revenue"],
    df_ratio["receivables"],
    df_ratio["revenue"].notna() & df_ratio["receivables_denominator_ok"]
)

ratio_cols = [
    "profit_margin",
    "roa",
    "roe",
    "equity_ratio_calc",
    "debt_ratio",
    "cash_to_assets",
    "receivables_to_assets",
    "revenue_per_employee_calc",
    "profit_per_employee",
    "asset_turnover",
    "debt_to_equity_ratio",
    "cash_ratio",
    "receivables_turnover"
]

df_ratio[ratio_cols].head()

,profit_margin,roa,roe,equity_ratio_calc,debt_ratio,cash_to_assets,receivables_to_assets,revenue_per_employee_calc,profit_per_employee,asset_turnover,debt_to_equity_ratio,cash_ratio,receivables_turnover
0,-0.164487,-0.166957,-0.567301,0.294300,0.526800,0.221900,0.087710,NaN,NaN,1.015014,1.790009,0.421222,11.572445
1,-0.050544,-0.319806,-0.350891,0.911410,0.043284,0.490663,0.487475,NaN,NaN,6.327315,0.047491,11.335977,12.979767
2,0.121203,0.196191,0.850419,0.230699,0.239036,0.625901,0.267640,91666.666667,11110.258333,1.618697,1.036136,2.618445,6.048035
3,0.196372,0.317880,0.431139,0.737302,0.201569,0.911394,0.073514,NaN,NaN,1.618767,0.273388,4.521494,22.019871
4,0.002040,0.009610,0.012224,0.786208,0.165515,0.239374,0.282284,246153.846154,502.260769,4.709898,0.210523,1.446239,16.684938


**Cell 71 - ratio availability summary**

In [83]:
ratio_availability_summary = pd.DataFrame({
    "ratio": ratio_cols,
    "non_missing": [df_ratio[col].notna().sum() for col in ratio_cols],
    "missing": [df_ratio[col].isna().sum() for col in ratio_cols],
    "non_missing_pct": [round(df_ratio[col].notna().mean() * 100, 2) for col in ratio_cols]
}).sort_values("non_missing", ascending=False)

ratio_availability_summary

,ratio,non_missing,missing,non_missing_pct
9,asset_turnover,91476,141,99.85
3,equity_ratio_calc,90247,1370,98.50
4,debt_ratio,89484,2133,97.67
10,debt_to_equity_ratio,81074,10543,88.49
6,receivables_to_assets,78529,13088,85.71
5,cash_to_assets,78359,13258,85.53
1,roa,78139,13478,85.29
0,profit_margin,77983,13634,85.12
12,receivables_turnover,76249,15368,83.23
11,cash_ratio,75593,16024,82.51


**Cell 72 - Raw ratio descriptive summary before outlier treatment**

In [84]:
raw_ratio_summary = df_ratio[ratio_cols].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
).T

raw_ratio_summary

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
profit_margin,77983.0,4.470881e-01,7.574854e+01,-2.089786e+03,-0.249651,-0.055944,0.000000,0.010313,4.578275e-02,4.012367e-01,1.446581e+00,2.060371e+04
roa,78139.0,3.002492e-01,5.806376e+01,-4.083709e+02,-0.528946,-0.152949,0.000000,0.030910,1.127267e-01,3.747007e-01,8.529519e-01,1.621049e+04
roe,71634.0,1.133257e+00,1.417262e+02,-1.036679e+04,-2.616778,-0.354207,0.000000,0.093709,2.792525e-01,9.269127e-01,3.777334e+00,3.515176e+04
equity_ratio_calc,90247.0,3.885196e-01,7.604107e+00,-2.263840e+03,-0.700447,-0.069049,0.135290,0.433546,7.291577e-01,9.660722e-01,9.979242e-01,1.000000e+00
debt_ratio,89484.0,5.740394e-01,1.178716e+01,0.000000e+00,0.000003,0.004878,0.106258,0.370035,7.502050e-01,9.868074e-01,1.000000e+00,2.264840e+03
cash_to_assets,78359.0,2.253980e-01,2.439647e-01,0.000000e+00,0.000009,0.000569,0.025254,0.134602,3.584732e-01,7.430545e-01,9.729423e-01,1.000000e+00
receivables_to_assets,78529.0,2.708010e-01,2.503338e-01,0.000000e+00,0.000000,0.002841,0.078238,0.196086,3.934368e-01,8.279466e-01,9.926438e-01,1.000000e+00
revenue_per_employee_calc,30254.0,1.950386e+07,1.399885e+09,0.000000e+00,14250.375940,57500.000000,170588.235294,366359.091805,1.175000e+06,2.700000e+07,1.200000e+08,2.357744e+11
profit_per_employee,28443.0,-3.982155e+04,3.163932e+07,-5.180488e+09,-486731.661182,-36758.212773,0.000000,4503.136667,2.155931e+04,4.032769e+05,3.236018e+06,4.812990e+08
asset_turnover,91476.0,1.845625e+02,5.078945e+04,0.000000e+00,0.086654,0.267875,1.366460,2.247268,4.261737e+00,1.073337e+01,2.761569e+01,1.533611e+07


### **Part 6 - Treat extreme and feasible ratio values**

**Cell 73 - Define ratio plausibility limits**

In [85]:
# ============================================================
# Ratio plausibility limits
# ============================================================
# These are broad limits, not strict accounting truths.
# Their purpose is to remove values that are too extreme to be useful
# for cross-sectional thesis analysis.
#
# Example:
# profit_margin = 50 means profit is 50 times revenue.
# This is usually caused by low revenue, special accounting effects,
# or data problems. It would dominate the analysis.
# ============================================================

ratio_plausibility_limits = {
    # Profitability ratios
    # Allow losses up to -200% and profits up to +200% of revenue.
    "profit_margin": (-2, 2),

    # ROA: profit relative to total assets.
    # Very large absolute values usually indicate unstable denominators or special cases.
    "roa": (-2, 2),

    # ROE can naturally be more volatile.
    # Still, values beyond +/-500% are too unstable for main analysis.
    "roe": (-5, 5),

    # Equity ratio can be negative if equity is negative.
    # Upper bound should not exceed 1 after our cleaning.
    "equity_ratio_calc": (-5, 1),

    # Debt ratio can exceed 1 when equity is negative.
    # But values above 5 are extreme distress/data problems.
    "debt_ratio": (0, 5),

    # Cash and receivables are asset components.
    "cash_to_assets": (0, 1),
    "receivables_to_assets": (0, 1),

    # Per employee values are not bounded here.
    # They will be handled by winsorization.
    "revenue_per_employee_calc": (0, np.inf),
    "profit_per_employee": (-np.inf, np.inf),

    # Asset turnover above 50 is usually not useful for stable cross-sectional analysis.
    "asset_turnover": (0, 50),

    # Debt-to-equity can be high when equity is small.
    # We already require equity >= 1000, but still cap infeasible extremes.
    "debt_to_equity_ratio": (0, 50),

    # Cash-to-liabilities proxy can be high if liabilities are low.
    "cash_ratio": (0, 100),

    # Receivables turnover can be high, but very high values are unstable.
    "receivables_turnover": (0, 1000)
}

ratio_plausibility_limits

{'profit_margin': (-2, 2),
 'roa': (-2, 2),
 'roe': (-5, 5),
 'equity_ratio_calc': (-5, 1),
 'debt_ratio': (0, 5),
 'cash_to_assets': (0, 1),
 'receivables_to_assets': (0, 1),
 'revenue_per_employee_calc': (0, inf),
 'profit_per_employee': (-inf, inf),
 'asset_turnover': (0, 50),
 'debt_to_equity_ratio': (0, 50),
 'cash_ratio': (0, 100),
 'receivables_turnover': (0, 1000)}

**Part 74 - Apply plausibility limits and create cleaned ratio columns**

In [86]:
ratio_cleaning_log = []

for col in ratio_cols:
    lower, upper = ratio_plausibility_limits[col]

    clean_col = col + "_clean"
    flag_col = col + "_outside_plausible_range"

    df_ratio[clean_col] = df_ratio[col].copy()

    outside_mask = (
        df_ratio[clean_col].notna() &
        (
            (df_ratio[clean_col] < lower) |
            (df_ratio[clean_col] > upper)
        )
    )

    df_ratio[flag_col] = outside_mask

    affected_count = int(outside_mask.sum())

    ratio_cleaning_log.append({
        "ratio": col,
        "clean_ratio": clean_col,
        "lower_limit": lower,
        "upper_limit": upper,
        "values_set_to_missing": affected_count,
        "values_set_to_missing_pct_of_non_missing": round(
            affected_count / max(df_ratio[col].notna().sum(), 1) * 100, 2
        )
    })

    df_ratio.loc[outside_mask, clean_col] = np.nan

ratio_cleaning_log_df = pd.DataFrame(ratio_cleaning_log)

ratio_cleaning_log_df

,ratio,clean_ratio,lower_limit,upper_limit,values_set_to_missing,values_set_to_missing_pct_of_non_missing
0,profit_margin,profit_margin_clean,-2.0,2.0,646,0.83
1,roa,roa_clean,-2.0,2.0,267,0.34
2,roe,roe_clean,-5.0,5.0,1016,1.42
3,equity_ratio_calc,equity_ratio_calc_clean,-5.0,1.0,28,0.03
4,debt_ratio,debt_ratio_clean,0.0,5.0,58,0.06
5,cash_to_assets,cash_to_assets_clean,0.0,1.0,0,0.00
6,receivables_to_assets,receivables_to_assets_clean,0.0,1.0,0,0.00
7,revenue_per_employee_calc,revenue_per_employee_calc_clean,0.0,inf,0,0.00
8,profit_per_employee,profit_per_employee_clean,-inf,inf,0,0.00
9,asset_turnover,asset_turnover_clean,0.0,50.0,323,0.35


**Cell 75 - Summary after hard plausibliilty cleaning**

In [87]:
ratio_clean_cols = [col + "_clean" for col in ratio_cols]

ratio_clean_summary = df_ratio[ratio_clean_cols].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
).T

ratio_clean_summary

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
profit_margin_clean,77337.0,5.162135e-02,1.848335e-01,-1.948406e+00,-0.234495,-0.055261,0.000000,0.010066,4.443418e-02,3.666495e-01,8.859138e-01,1.993954e+00
roa_clean,77872.0,6.340993e-02,1.964887e-01,-1.992336e+00,-0.504184,-0.150587,0.000000,0.030767,1.120035e-01,3.633770e-01,7.767904e-01,1.981112e+00
roe_clean,70618.0,1.563532e-01,5.295098e-01,-4.968391e+00,-1.590818,-0.304506,0.000000,0.092901,2.738184e-01,8.737468e-01,1.858182e+00,4.968689e+00
equity_ratio_calc_clean,90219.0,4.222952e-01,3.733882e-01,-4.413298e+00,-0.690913,-0.067666,0.135478,0.433735,7.292307e-01,9.660926e-01,9.979287e-01,1.000000e+00
debt_ratio_clean,89426.0,4.337801e-01,3.502108e-01,0.000000e+00,0.000003,0.004869,0.106173,0.369565,7.495579e-01,9.864575e-01,9.998652e-01,4.932904e+00
cash_to_assets_clean,78359.0,2.253980e-01,2.439647e-01,0.000000e+00,0.000009,0.000569,0.025254,0.134602,3.584732e-01,7.430545e-01,9.729423e-01,1.000000e+00
receivables_to_assets_clean,78529.0,2.708010e-01,2.503338e-01,0.000000e+00,0.000000,0.002841,0.078238,0.196086,3.934368e-01,8.279466e-01,9.926438e-01,1.000000e+00
revenue_per_employee_calc_clean,30254.0,1.950386e+07,1.399885e+09,0.000000e+00,14250.375940,57500.000000,170588.235294,366359.091805,1.175000e+06,2.700000e+07,1.200000e+08,2.357744e+11
profit_per_employee_clean,28443.0,-3.982155e+04,3.163932e+07,-5.180488e+09,-486731.661182,-36758.212773,0.000000,4503.136667,2.155931e+04,4.032769e+05,3.236018e+06,4.812990e+08
asset_turnover_clean,91153.0,3.531331e+00,4.312496e+00,0.000000e+00,0.086012,0.267202,1.362118,2.241397,4.226231e+00,1.034445e+01,2.318940e+01,4.994536e+01


**Cell 76 - Winsorization function**

In [88]:
def winsorize_series(series, lower_q=0.01, upper_q=0.99):
    """
    Winsorizes a series by capping values at selected quantiles.

    Example:
    lower_q = 0.01 means values below the 1st percentile
    are set to the 1st percentile.

    upper_q = 0.99 means values above the 99th percentile
    are set to the 99th percentile.

    This reduces the influence of extreme values without deleting observations.
    """

    s = series.copy()

    if s.notna().sum() < 50:
        return s

    lower = s.quantile(lower_q)
    upper = s.quantile(upper_q)

    return s.clip(lower=lower, upper=upper)

**Cell 77 - Apply winsorization to cleaned ratios**

In [89]:
winsorized_ratio_cols = []

for col in ratio_clean_cols:
    win_col = col.replace("_clean", "_win")

    df_ratio[win_col] = winsorize_series(
        df_ratio[col],
        lower_q=0.01,
        upper_q=0.99
    )

    winsorized_ratio_cols.append(win_col)

winsorized_ratio_cols

['profit_margin_win',
 'roa_win',
 'roe_win',
 'equity_ratio_calc_win',
 'debt_ratio_win',
 'cash_to_assets_win',
 'receivables_to_assets_win',
 'revenue_per_employee_calc_win',
 'profit_per_employee_win',
 'asset_turnover_win',
 'debt_to_equity_ratio_win',
 'cash_ratio_win',
 'receivables_turnover_win']

**Cell 78 - Final winsorized ratio summary**

In [90]:
winsorized_ratio_summary = df_ratio[winsorized_ratio_cols].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
).T

winsorized_ratio_summary

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
profit_margin_win,77337.0,5.031616e-02,1.499318e-01,-0.234495,-0.234394,-0.055261,0.000000,0.010066,4.443418e-02,3.666495e-01,8.858536e-01,8.859138e-01
roa_win,77872.0,6.396299e-02,1.693469e-01,-0.504184,-0.504130,-0.150587,0.000000,0.030767,1.120035e-01,3.633770e-01,7.766904e-01,7.767904e-01
roe_win,70618.0,1.575474e-01,4.128740e-01,-1.590818,-1.590118,-0.304506,0.000000,0.092901,2.738184e-01,8.737468e-01,1.858114e+00,1.858182e+00
equity_ratio_calc_win,90219.0,4.248749e-01,3.619278e-01,-0.690913,-0.690906,-0.067666,0.135478,0.433735,7.292307e-01,9.660926e-01,9.979263e-01,9.979287e-01
debt_ratio_win,89426.0,4.314139e-01,3.401542e-01,0.000003,0.000003,0.004869,0.106173,0.369565,7.495579e-01,9.864575e-01,9.998642e-01,9.998652e-01
cash_to_assets_win,78359.0,2.251907e-01,2.433179e-01,0.000009,0.000009,0.000569,0.025254,0.134602,3.584732e-01,7.430545e-01,9.729156e-01,9.729423e-01
receivables_to_assets_win,78529.0,2.707430e-01,2.501656e-01,0.000000,0.000000,0.002841,0.078238,0.196086,3.934368e-01,8.279466e-01,9.926401e-01,9.926438e-01
revenue_per_employee_calc_win,30254.0,5.308693e+06,1.663832e+07,14250.375940,14269.105263,57500.000000,170588.235294,366359.091805,1.175000e+06,2.700000e+07,1.200000e+08,1.200000e+08
profit_per_employee_win,28443.0,8.553366e+04,4.074024e+05,-486731.661182,-485792.093767,-36758.212773,0.000000,4503.136667,2.155931e+04,4.032769e+05,3.229313e+06,3.236018e+06
asset_turnover_win,91153.0,3.445404e+00,3.737976e+00,0.086012,0.086025,0.267202,1.362118,2.241397,4.226231e+00,1.034445e+01,2.318571e+01,2.318940e+01


**Cell 79 - Compare raw, clean, and winsorised availability**

In [93]:
ratio_final_quality = []

for ratio in ratio_cols:
    raw_col = ratio
    clean_col = ratio + "_clean"
    win_col = ratio + "_win"

    ratio_final_quality.append({
        "ratio": ratio,
        "raw_non_missing": df_ratio[raw_col].notna().sum(),
        "clean_non_missing": df_ratio[clean_col].notna().sum(),
        "winsorized_non_missing": df_ratio[win_col].notna().sum(),
        "lost_from_plausibility_rules": df_ratio[raw_col].notna().sum() - df_ratio[clean_col].notna().sum(),
        "final_usable_pct": round(df_ratio[win_col].notna().mean() * 100, 2)
    })

ratio_final_quality_df = pd.DataFrame(ratio_final_quality).sort_values(
    "winsorized_non_missing",
    ascending=False
)

ratio_final_quality_df

,ratio,raw_non_missing,clean_non_missing,winsorized_non_missing,lost_from_plausibility_rules,final_usable_pct
9,asset_turnover,91476,91153,91153,323,99.49
3,equity_ratio_calc,90247,90219,90219,28,98.47
4,debt_ratio,89484,89426,89426,58,97.61
6,receivables_to_assets,78529,78529,78529,0,85.71
5,cash_to_assets,78359,78359,78359,0,85.53
10,debt_to_equity_ratio,81074,77877,77877,3197,85.00
1,roa,78139,77872,77872,267,85.00
0,profit_margin,77983,77337,77337,646,84.41
12,receivables_turnover,76249,75751,75751,498,82.68
11,cash_ratio,75593,74714,74714,879,81.55


**Cell 81 - Check top extreme values after winsorization**

In [94]:
def show_ratio_extremes(df, ratio_col, n=10):
    display_cols = [
        "Name",
        "financial_year",
        "wz_4digit",
        "wz_section",
        "status_group",
        ratio_col
    ]

    display_cols = [c for c in display_cols if c in df.columns]

    print("\n" + "="*90)
    print(f"Ratio: {ratio_col}")
    print("="*90)

    print("\nSmallest values:")
    display(
        df[df[ratio_col].notna()]
        .sort_values(ratio_col, ascending=True)
        [display_cols]
        .head(n)
    )

    print("\nLargest values:")
    display(
        df[df[ratio_col].notna()]
        .sort_values(ratio_col, ascending=False)
        [display_cols]
        .head(n)
    )


for col in winsorized_ratio_cols:
    show_ratio_extremes(df_ratio, col, n=5)


Ratio: profit_margin_win

Smallest values:


,Name,financial_year,wz_4digit,wz_section,status_group,profit_margin_win
65330,KSL-Kuttler Automation Systems GmbH,2022.0,28.99,C Manufacturing,active,-0.234495
41231,HeckTec GmbH,2020.0,28.99,C Manufacturing,inactive_or_distress,-0.234495
16417,mailo AG,2023.0,66.22,K Financial and insurance activities,active,-0.234495
67856,ESV Schwenger GmbH & Co. KG,2023.0,66.22,K Financial and insurance activities,active,-0.234495
30658,HTB 11. Geschlossene Immobilieninvestment Port...,2024.0,68.31,L Real estate activities,active,-0.234495



Largest values:


,Name,financial_year,wz_4digit,wz_section,status_group,profit_margin_win
29111,Das Grüne Emissionshaus GmbH,2023.0,35.11,D Energy supply,active,0.885914
30680,Lübke Kelber AG,2023.0,68.31,L Real estate activities,active,0.885914
30525,MGR Holding GmbH,2023.0,68.31,L Real estate activities,active,0.885914
46439,ASN Immobilien GmbH,2024.0,68.10,L Real estate activities,active,0.885914
29231,Immodera GmbH,2023.0,68.10,L Real estate activities,active,0.885914



Ratio: roa_win

Smallest values:


...

,Name,financial_year,wz_4digit,wz_section,status_group,cash_ratio_win
98800,Nord-Immopool UG,2021.0,68.31,L Real estate activities,other_or_unknown,49.206753
5785,BVT-CAM Private Equity New Markets Fund GmbH &...,2023.0,66.19,K Financial and insurance activities,inactive_or_distress,49.206753
5819,HAFA Vermögensverwaltungs GmbH,2021.0,66.19,K Financial and insurance activities,active,49.206753
98110,ßeta GmbH Bakery Engineering,2021.0,28.22,C Manufacturing,inactive_or_distress,49.206753
6428,Aon Beteiligungsmanagement Deutschland GmbH & ...,2023.0,66.22,K Financial and insurance activities,active,49.206753



Ratio: receivables_turnover_win

Smallest values:


,Name,financial_year,wz_4digit,wz_section,status_group,receivables_turnover_win
77655,Jung Verwaltung GmbH,2022.0,68.31,L Real estate activities,active,0.674089
46119,RENERTEC GmbH Gesellschaft für regenerative un...,2024.0,35.11,D Energy supply,active,0.674089
56864,Best choice Autoteile UG,2021.0,45.31,G Wholesale and retail trade,active,0.674089
32329,Windkraft Reichenbach II GmbH & Co. KG,2024.0,35.11,D Energy supply,active,0.674089
77430,Novatex Immobilienverwaltungs-GmbH,2022.0,68.31,L Real estate activities,active,0.674089



Largest values:


,Name,financial_year,wz_4digit,wz_section,status_group,receivables_turnover_win
98090,Maschinenbau Karl Ruhl GmbH,2022.0,28.41,C Manufacturing,inactive_or_distress,387.898438
5,Solardach Walternienburg GmbH & Co. KG,2022.0,71.12,"M Professional, scientific and technical activ...",active,387.898438
10,Lohlein Ingenieur GmbH,2020.0,71.12,"M Professional, scientific and technical activ...",active,387.898438
5522,Kirchner Vermögensverwaltungs GmbH,2023.0,66.19,K Financial and insurance activities,active,387.898438
5553,Anoria Invest 1 GmbH,2024.0,66.19,K Financial and insurance activities,active,387.898438


### **Part 7- Create final thesis-ready dataset**

**Cell 81 - Add log transformation for size variables**

In [95]:
# ============================================================
# Log variables
# ============================================================
# Financial variables are highly skewed.
# Logs make them more stable for regression and plots.
#
# log1p(x) = log(1 + x)
# This works safely when x = 0.
# ============================================================

df_ratio["log_assets"] = np.where(
    df_ratio["assets"].notna() & (df_ratio["assets"] >= 0),
    np.log1p(df_ratio["assets"]),
    np.nan
)

df_ratio["log_revenue"] = np.where(
    df_ratio["revenue"].notna() & (df_ratio["revenue"] >= 0),
    np.log1p(df_ratio["revenue"]),
    np.nan
)

df_ratio["log_employees"] = np.where(
    df_ratio["employees"].notna() & (df_ratio["employees"] >= 1),
    np.log1p(df_ratio["employees"]),
    np.nan
)

# Profit can be negative, so normal log is not possible.
# Signed log keeps the sign and reduces scale.
df_ratio["signed_log_profit"] = np.where(
    df_ratio["profit"].notna(),
    np.sign(df_ratio["profit"]) * np.log1p(np.abs(df_ratio["profit"])),
    np.nan
)

df_ratio[["log_assets", "log_revenue", "log_employees", "signed_log_profit"]].describe().T

,count,mean,std,min,25%,50%,75%,max
log_assets,91617.0,14.165125,1.929168,0.693147,13.097340,14.045655,15.321680,28.700514
log_revenue,91590.0,14.926100,1.951149,0.000000,13.997833,15.009433,16.045525,26.879288
log_employees,30266.0,2.636717,1.216675,0.693147,1.791759,2.564949,3.367296,10.736418
signed_log_profit,78200.0,5.928768,9.096163,-25.258537,0.000000,10.518039,12.303118,23.124685


**Cell 82 - Create final column list**

In [96]:
final_ratio_cols = winsorized_ratio_cols

final_analysis_cols = [
    # Identifiers
    "company_key",
    "Name",
    "Register-ID",
    "North Data URL",

    # Time and status
    "financial_date",
    "financial_year",
    "status_group",

    # Industry variables
    "industry_raw",
    "industry_label",
    "wz_code_detailed",
    "wz_4digit",
    "wz_2digit",
    "wz_section",
    "is_financial_sector",
    "is_real_estate_sector",
    "is_manufacturing_sector",

    # Clean financial base variables
    "assets",
    "profit",
    "revenue",
    "equity",
    "liabilities",
    "cash",
    "receivables",
    "employees",

    # Log variables
    "log_assets",
    "log_revenue",
    "log_employees",
    "signed_log_profit",

    # Final winsorized ratios
] + final_ratio_cols

final_analysis_cols = [col for col in final_analysis_cols if col in df_ratio.columns]

df_final = df_ratio[final_analysis_cols].copy()

print("Final analysis dataset shape:", df_final.shape)
df_final.head()

Final analysis dataset shape: (91617, 41)


,company_key,Name,Register-ID,North Data URL,financial_date,financial_year,status_group,industry_raw,industry_label,wz_code_detailed,...,equity_ratio_calc_win,debt_ratio_win,cash_to_assets_win,receivables_to_assets_win,revenue_per_employee_calc_win,profit_per_employee_win,asset_turnover_win,debt_to_equity_ratio_win,cash_ratio_win,receivables_turnover_win
0,https://www.northdata.de/Knut%20R%C3%B6sch%20B...,Knut Rösch Baugrunduntersuchungen GmbH,HRB 7129 KI,https://www.northdata.de/Knut%20R%C3%B6sch%20B...,2024-12-31,2024.0,active,71.12.9 Sonstige Ingenieurbüros,Sonstige Ingenieurbüros,71.12.9,...,0.294300,0.526800,0.221900,0.087710,NaN,NaN,1.015014,1.790009,0.421222,11.572445
1,https://www.northdata.de/Soulworks%20Developme...,Soulworks Developments GmbH,HRB 288800,https://www.northdata.de/Soulworks%20Developme...,2022-12-31,2022.0,active,71.11.1 Architekturbüros für Hochbau,Architekturbüros für Hochbau,71.11.1,...,0.911410,0.043284,0.490663,0.487475,NaN,NaN,6.327315,0.047491,11.335977,12.979767
2,https://www.northdata.de/BEHR%20INGENIEURE%20G...,BEHR INGENIEURE GmbH,HRB 11586,https://www.northdata.de/BEHR%20INGENIEURE%20G...,2023-12-31,2023.0,active,71.12.1 Ingenieurbüros für bautechnische Gesam...,Ingenieurbüros für bautechnische Gesamtplanung,71.12.1,...,0.230699,0.239036,0.625901,0.267640,91666.666667,11110.258333,1.618697,1.036136,2.618445,6.048035
3,https://www.northdata.de/GLASFAKTOR%20Ingenieu...,GLASFAKTOR Ingenieure GmbH,HRB 30711,https://www.northdata.de/GLASFAKTOR%20Ingenieu...,2023-11-30,2023.0,active,71.12.1 Ingenieurbüros für bautechnische Gesam...,Ingenieurbüros für bautechnische Gesamtplanung,71.12.1,...,0.737302,0.201569,0.911394,0.073514,NaN,NaN,1.618767,0.273388,4.521494,22.019871
4,https://www.northdata.de/Novum%20Analytik%20Gm...,Novum Analytik GmbH,HRB 723862,https://www.northdata.de/Novum%20Analytik%20Gm...,2023-12-31,2023.0,active,"71.20.0 Technische, physikalische und chemisch...","Technische, physikalische und chemische Unters...",71.20.0,...,0.786208,0.165515,0.239374,0.282284,246153.846154,502.260769,4.709898,0.210523,1.446239,16.684938


**Cell 83 - Final missingness summary**

In [97]:
final_missing_summary = pd.DataFrame({
    "column": df_final.columns,
    "non_missing": [df_final[col].notna().sum() for col in df_final.columns],
    "missing": [df_final[col].isna().sum() for col in df_final.columns],
    "missing_pct": [round(df_final[col].isna().mean() * 100, 2) for col in df_final.columns]
}).sort_values("missing_pct", ascending=False)

final_missing_summary

,column,non_missing,missing,missing_pct
36,profit_per_employee_win,28443,63174,68.95
35,revenue_per_employee_calc_win,30254,61363,66.98
23,employees,30266,61351,66.96
26,log_employees,30266,61351,66.96
30,roe_win,70618,20999,22.92
39,cash_ratio_win,74714,16903,18.45
40,receivables_turnover_win,75751,15866,17.32
28,profit_margin_win,77337,14280,15.59
29,roa_win,77872,13745,15.00
38,debt_to_equity_ratio_win,77877,13740,15.00


How we are tackling extreme outliers

Extreme financial observations were handled in multiple steps. First, clearly impossible values, such as negative revenue, negative cash, negative receivables, or cash exceeding total assets, were set to missing. Second, ratios were only calculated when the denominator was sufficiently large to avoid unstable divisions. Third, economically implausible ratio values were set to missing using broad plausibility limits. Finally, the remaining ratio variables were winsorized at the 1st and 99th percentiles to reduce the influence of extreme observations while preserving the sample size.

**Cell 84 - Save final thesis-ready datasets**

In [103]:
# Main final dataset
final_dataset_path = output_folder / "DE_thesis_analysis_final_ratio_ready.csv"

# Full dataset with raw, clean, and winsorized ratios
full_ratio_dataset_path = output_folder / "DE_thesis_analysis_full_ratio_versions.csv"

# Summary tables
raw_ratio_summary_path = output_folder / "part5_raw_ratio_summary.csv"
ratio_cleaning_log_path = output_folder / "part6_ratio_cleaning_log.csv"
ratio_clean_summary_path = output_folder / "part6_ratio_clean_summary.csv"
winsorized_ratio_summary_path = output_folder / "part6_winsorized_ratio_summary.csv"
ratio_final_quality_path = output_folder / "part6_ratio_final_quality.csv"
final_missing_summary_path = output_folder / "part7_final_missing_summary.csv"

df_final.to_csv(final_dataset_path, index=False, encoding="utf-8-sig")
df_ratio.to_csv(full_ratio_dataset_path, index=False, encoding="utf-8-sig")

raw_ratio_summary.to_csv(raw_ratio_summary_path, index=True, encoding="utf-8-sig")
ratio_cleaning_log_df.to_csv(ratio_cleaning_log_path, index=False, encoding="utf-8-sig")
ratio_clean_summary.to_csv(ratio_clean_summary_path, index=True, encoding="utf-8-sig")
winsorized_ratio_summary.to_csv(winsorized_ratio_summary_path, index=True, encoding="utf-8-sig")
ratio_final_quality_df.to_csv(ratio_final_quality_path, index=False, encoding="utf-8-sig")
final_missing_summary.to_csv(final_missing_summary_path, index=False, encoding="utf-8-sig")

print("Saved final thesis-ready dataset:")
print(final_dataset_path)

print("\nSaved full ratio-version dataset:")
print(full_ratio_dataset_path)

print("\nSaved summary files in:")
print(output_folder)

print("\nFinal dataset shape:", df_final.shape)

Saved final thesis-ready dataset:
/content/drive/MyDrive/IBS_7_Thesis/Python Files/cleaning_audit_outputs/DE_thesis_analysis_final_ratio_ready.csv

Saved full ratio-version dataset:
/content/drive/MyDrive/IBS_7_Thesis/Python Files/cleaning_audit_outputs/DE_thesis_analysis_full_ratio_versions.csv

Saved summary files in:
/content/drive/MyDrive/IBS_7_Thesis/Python Files/cleaning_audit_outputs

Final dataset shape: (91617, 41)


In [98]:
print("Final dataset shape:", df_final.shape)

print("\nYear distribution:")
display(df_final["financial_year"].value_counts().sort_index().to_frame("rows"))

print("\nStatus distribution:")
display(df_final["status_group"].value_counts(dropna=False).to_frame("rows"))

print("\nBroad sector distribution:")
display(df_final["wz_section"].value_counts(dropna=False).to_frame("rows"))

print("\nFinal ratio quality:")
display(ratio_final_quality_df)

Final dataset shape: (91617, 41)

Year distribution:


,rows
financial_year,
2020.0,4742
2021.0,6718
2022.0,9109
2023.0,54910
2024.0,16138



Status distribution:


,rows
status_group,
active,89952
inactive_or_distress,1624
other_or_unknown,41



Broad sector distribution:


,rows
wz_section,
C Manufacturing,35810
K Financial and insurance activities,14973
"M Professional, scientific and technical activities",13627
G Wholesale and retail trade,11796
L Real estate activities,8918
N Administrative and support service activities,3068
F Construction,1003
D Energy supply,744
"E Water, sewerage and waste",677



Final ratio quality:


,ratio,raw_non_missing,clean_non_missing,winsorized_non_missing,lost_from_plausibility_rules,final_usable_pct
9,asset_turnover,91476,91153,91153,323,99.49
3,equity_ratio_calc,90247,90219,90219,28,98.47
4,debt_ratio,89484,89426,89426,58,97.61
6,receivables_to_assets,78529,78529,78529,0,85.71
5,cash_to_assets,78359,78359,78359,0,85.53
10,debt_to_equity_ratio,81074,77877,77877,3197,85.00
1,roa,78139,77872,77872,267,85.00
0,profit_margin,77983,77337,77337,646,84.41
12,receivables_turnover,76249,75751,75751,498,82.68
11,cash_ratio,75593,74714,74714,879,81.55


### **Part 8 - Thesis readiness and stabillity checks**

**Cell 86 - Start thesis-readiness check**

In [104]:
# ============================================================
# PART 8: THESIS READINESS AND STABILITY CHECKS
# ============================================================
# Goal:
# Check whether the final dataset is stable enough for thesis analysis.
#
# We check:
# 1. Final sample structure
# 2. Ratio availability
# 3. Sector coverage
# 4. Year coverage
# 5. Financial-sector sensitivity
# 6. Active vs inactive/distress comparison
# ============================================================

df_analysis = df_final.copy()

print("Final analysis dataset shape:", df_analysis.shape)

Final analysis dataset shape: (91617, 41)


**Cell 87 - Confirm final ratio column names**

In [105]:
final_ratio_cols = [
    "profit_margin_win",
    "roa_win",
    "roe_win",
    "equity_ratio_calc_win",
    "debt_ratio_win",
    "cash_to_assets_win",
    "receivables_to_assets_win",
    "revenue_per_employee_calc_win",
    "profit_per_employee_win",
    "asset_turnover_win",
    "debt_to_equity_ratio_win",
    "cash_ratio_win",
    "receivables_turnover_win"
]

final_ratio_cols = [col for col in final_ratio_cols if col in df_analysis.columns]

print("Final ratio columns available:")
for col in final_ratio_cols:
    print("-", col)

Final ratio columns available:
- profit_margin_win
- roa_win
- roe_win
- equity_ratio_calc_win
- debt_ratio_win
- cash_to_assets_win
- receivables_to_assets_win
- revenue_per_employee_calc_win
- profit_per_employee_win
- asset_turnover_win
- debt_to_equity_ratio_win
- cash_ratio_win
- receivables_turnover_win


**Cell 88 - final thesis readiness summary**

In [106]:
readiness_summary = pd.DataFrame({
    "item": [
        "Total observations",
        "Unique companies",
        "Years covered",
        "Broad sectors",
        "WZ 4-digit industries",
        "Active firms",
        "Inactive/distress firms",
        "Financial-sector firms",
        "Non-financial-sector firms"
    ],
    "value": [
        len(df_analysis),
        df_analysis["company_key"].nunique(dropna=True),
        sorted(df_analysis["financial_year"].dropna().unique().tolist()),
        df_analysis["wz_section"].nunique(dropna=True),
        df_analysis["wz_4digit"].nunique(dropna=True),
        (df_analysis["status_group"] == "active").sum(),
        (df_analysis["status_group"] == "inactive_or_distress").sum(),
        df_analysis["is_financial_sector"].sum(),
        (~df_analysis["is_financial_sector"]).sum()
    ]
})

readiness_summary

,item,value
0,Total observations,91617
1,Unique companies,90261
2,Years covered,"[2020.0, 2021.0, 2022.0, 2023.0, 2024.0]"
3,Broad sectors,19
4,WZ 4-digit industries,428
5,Active firms,89952
6,Inactive/distress firms,1624
7,Financial-sector firms,14973
8,Non-financial-sector firms,76644


**Cell 89 - Sector-level sample size**

In [107]:
sector_sample_size = (
    df_analysis
    .groupby("wz_section", dropna=False)
    .agg(
        rows=("company_key", "size"),
        unique_companies=("company_key", "nunique")
    )
    .reset_index()
    .sort_values("rows", ascending=False)
)

sector_sample_size

,wz_section,rows,unique_companies
2,C Manufacturing,35810,35676
10,K Financial and insurance activities,14973,14662
12,"M Professional, scientific and technical activ...",13627,13606
6,G Wholesale and retail trade,11796,11201
11,L Real estate activities,8918,8704
13,N Administrative and support service activities,3068,3033
5,F Construction,1003,984
3,D Energy supply,744,744
4,"E Water, sewerage and waste",677,662
9,J Information and communication,284,282


**Cell 90 - keep only sufficiently large sectors for main sector comparison**

In [108]:
MIN_SECTOR_ROWS = 500

large_sectors = (
    sector_sample_size
    .loc[sector_sample_size["rows"] >= MIN_SECTOR_ROWS, "wz_section"]
    .tolist()
)

df_large_sector = df_analysis[df_analysis["wz_section"].isin(large_sectors)].copy()

print("Number of large sectors:", len(large_sectors))
print("Rows in large-sector dataset:", len(df_large_sector))
print("Rows excluded from small sectors:", len(df_analysis) - len(df_large_sector))

large_sectors

Number of large sectors: 9
Rows in large-sector dataset: 90616
Rows excluded from small sectors: 1001


['C Manufacturing',
 'K Financial and insurance activities',
 'M Professional, scientific and technical activities',
 'G Wholesale and retail trade',
 'L Real estate activities',
 'N Administrative and support service activities',
 'F Construction',
 'D Energy supply',
 'E Water, sewerage and waste']

**Cell 91 - Sector-level median ratio table**

In [109]:
main_ratio_cols = [
    "profit_margin_win",
    "roa_win",
    "equity_ratio_calc_win",
    "debt_ratio_win",
    "cash_to_assets_win",
    "receivables_to_assets_win",
    "asset_turnover_win"
]

main_ratio_cols = [col for col in main_ratio_cols if col in df_analysis.columns]

sector_ratio_medians = (
    df_large_sector
    .groupby("wz_section")[main_ratio_cols]
    .median()
    .reset_index()
)

sector_ratio_counts = (
    df_large_sector
    .groupby("wz_section")[main_ratio_cols]
    .count()
    .reset_index()
)

sector_ratio_medians

,wz_section,profit_margin_win,roa_win,equity_ratio_calc_win,debt_ratio_win,cash_to_assets_win,receivables_to_assets_win,asset_turnover_win
0,C Manufacturing,0.009021,0.032856,0.454019,0.360862,0.163013,0.211786,2.820562
1,D Energy supply,0.521145,0.127554,0.426997,0.222058,0.156591,0.123167,0.228247
2,"E Water, sewerage and waste",0.014468,0.045386,0.428091,0.354575,0.157245,0.210851,3.008270
3,F Construction,0.016767,0.049085,0.379373,0.442622,0.086280,0.167437,1.708683
4,G Wholesale and retail trade,0.008994,0.039797,0.387757,0.393065,0.118273,0.200093,3.307898
5,K Financial and insurance activities,0.017122,0.027953,0.541801,0.242834,0.067053,0.124800,1.457707
6,L Real estate activities,0.000000,0.000000,0.284921,0.585549,0.044253,0.055011,0.463053
7,"M Professional, scientific and technical activ...",0.014660,0.036007,0.429693,0.367377,0.231344,0.258511,1.799182
8,N Administrative and support service activities,0.007027,0.021345,0.378554,0.479683,0.102758,0.178718,2.134098


**Cell 92 - Add sector sample size to ratio table**

In [110]:
sector_ratio_table = (
    sector_sample_size
    .merge(sector_ratio_medians, on="wz_section", how="inner")
    .sort_values("rows", ascending=False)
)

sector_ratio_table

,wz_section,rows,unique_companies,profit_margin_win,roa_win,equity_ratio_calc_win,debt_ratio_win,cash_to_assets_win,receivables_to_assets_win,asset_turnover_win
0,C Manufacturing,35810,35676,0.009021,0.032856,0.454019,0.360862,0.163013,0.211786,2.820562
1,K Financial and insurance activities,14973,14662,0.017122,0.027953,0.541801,0.242834,0.067053,0.124800,1.457707
2,"M Professional, scientific and technical activ...",13627,13606,0.014660,0.036007,0.429693,0.367377,0.231344,0.258511,1.799182
3,G Wholesale and retail trade,11796,11201,0.008994,0.039797,0.387757,0.393065,0.118273,0.200093,3.307898
4,L Real estate activities,8918,8704,0.000000,0.000000,0.284921,0.585549,0.044253,0.055011,0.463053
5,N Administrative and support service activities,3068,3033,0.007027,0.021345,0.378554,0.479683,0.102758,0.178718,2.134098
6,F Construction,1003,984,0.016767,0.049085,0.379373,0.442622,0.086280,0.167437,1.708683
7,D Energy supply,744,744,0.521145,0.127554,0.426997,0.222058,0.156591,0.123167,0.228247
8,"E Water, sewerage and waste",677,662,0.014468,0.045386,0.428091,0.354575,0.157245,0.210851,3.008270


**Cell 93 - Active vs inactive/distress comparison**

In [111]:
status_ratio_table = (
    df_analysis
    .groupby("status_group")[main_ratio_cols]
    .median()
    .reset_index()
)

status_counts = (
    df_analysis
    .groupby("status_group")
    .agg(rows=("company_key", "size"))
    .reset_index()
)

status_ratio_table = status_counts.merge(
    status_ratio_table,
    on="status_group",
    how="left"
)

status_ratio_table

,status_group,rows,profit_margin_win,roa_win,equity_ratio_calc_win,debt_ratio_win,cash_to_assets_win,receivables_to_assets_win,asset_turnover_win
0,active,89952,0.010252,0.031217,0.435695,0.367880,0.134881,0.196116,2.240175
1,inactive_or_distress,1624,0.001229,0.004432,0.293485,0.559873,0.101837,0.195671,2.314815
2,other_or_unknown,41,-0.004218,-0.020833,0.069340,0.559712,0.258946,0.096959,2.533020


**Cell 94 - Financial sector sensitivity check**

In [112]:
df_nonfinancial = df_analysis[df_analysis["is_financial_sector"] == False].copy()
df_financial = df_analysis[df_analysis["is_financial_sector"] == True].copy()

financial_sensitivity_summary = pd.DataFrame({
    "dataset": [
        "All firms",
        "Financial sector only",
        "Excluding financial sector"
    ],
    "rows": [
        len(df_analysis),
        len(df_financial),
        len(df_nonfinancial)
    ],
    "profit_margin_median": [
        df_analysis["profit_margin_win"].median(),
        df_financial["profit_margin_win"].median(),
        df_nonfinancial["profit_margin_win"].median()
    ],
    "roa_median": [
        df_analysis["roa_win"].median(),
        df_financial["roa_win"].median(),
        df_nonfinancial["roa_win"].median()
    ],
    "equity_ratio_median": [
        df_analysis["equity_ratio_calc_win"].median(),
        df_financial["equity_ratio_calc_win"].median(),
        df_nonfinancial["equity_ratio_calc_win"].median()
    ],
    "debt_ratio_median": [
        df_analysis["debt_ratio_win"].median(),
        df_financial["debt_ratio_win"].median(),
        df_nonfinancial["debt_ratio_win"].median()
    ],
    "asset_turnover_median": [
        df_analysis["asset_turnover_win"].median(),
        df_financial["asset_turnover_win"].median(),
        df_nonfinancial["asset_turnover_win"].median()
    ]
})

financial_sensitivity_summary

,dataset,rows,profit_margin_median,roa_median,equity_ratio_median,debt_ratio_median,asset_turnover_median
0,All firms,91617,0.010066,0.030767,0.433735,0.369565,2.241397
1,Financial sector only,14973,0.017122,0.027953,0.541801,0.242834,1.457707
2,Excluding financial sector,76644,0.009104,0.031387,0.420258,0.387285,2.507793


**Cell 95 - Year stability check**

In [113]:
year_ratio_table = (
    df_analysis
    .groupby("financial_year")[main_ratio_cols]
    .median()
    .reset_index()
    .sort_values("financial_year")
)

year_counts = (
    df_analysis
    .groupby("financial_year")
    .agg(rows=("company_key", "size"))
    .reset_index()
)

year_ratio_table = year_counts.merge(
    year_ratio_table,
    on="financial_year",
    how="left"
)

year_ratio_table

,financial_year,rows,profit_margin_win,roa_win,equity_ratio_calc_win,debt_ratio_win,cash_to_assets_win,receivables_to_assets_win,asset_turnover_win
0,2020.0,4742,0.003200,0.012766,0.412850,0.392233,0.120874,0.211312,2.176491
1,2021.0,6718,0.004532,0.016910,0.423946,0.381529,0.115779,0.184768,2.020256
2,2022.0,9109,0.004673,0.015473,0.379445,0.430890,0.101575,0.205523,2.108513
3,2023.0,54910,0.010844,0.032680,0.428289,0.376383,0.134148,0.200359,2.300048
4,2024.0,16138,0.013925,0.039370,0.488829,0.316255,0.159512,0.179250,2.195039


**Cell 96 - Top WZ 4-digit industry comparsion**

In [114]:
MIN_WZ4_ROWS = 500

wz4_sample_size = (
    df_analysis
    .groupby(["wz_4digit", "industry_label"], dropna=False)
    .agg(rows=("company_key", "size"))
    .reset_index()
    .sort_values("rows", ascending=False)
)

large_wz4_codes = (
    wz4_sample_size
    .loc[wz4_sample_size["rows"] >= MIN_WZ4_ROWS, "wz_4digit"]
    .tolist()
)

df_large_wz4 = df_analysis[df_analysis["wz_4digit"].isin(large_wz4_codes)].copy()

wz4_ratio_table = (
    df_large_wz4
    .groupby(["wz_4digit", "industry_label"])[main_ratio_cols]
    .median()
    .reset_index()
    .merge(wz4_sample_size, on=["wz_4digit", "industry_label"], how="left")
    .sort_values("rows", ascending=False)
)

wz4_ratio_table.head(30)

,wz_4digit,industry_label,profit_margin_win,roa_win,equity_ratio_calc_win,debt_ratio_win,cash_to_assets_win,receivables_to_assets_win,asset_turnover_win,rows
53,66.19,Sonstige mit Finanzdienstleistungen verbundene...,0.017099,0.028315,0.590212,0.308488,0.046350,0.083549,1.405786,9571
57,68.31,"Vermittlung von Wohngrundstücken, Wohngebäuden...",0.000000,0.000000,0.248192,0.627894,0.042753,0.058304,0.511731,7755
54,66.22,Tätigkeit von Versicherungsmaklerinnen und -ma...,0.020975,0.030367,0.463891,0.248342,0.238188,0.292354,1.562222,3894
64,71.12,Ingenieurbüros für bautechnische Gesamtplanung,0.024258,0.036344,0.387109,0.425226,0.233462,0.255331,1.573252,3268
65,71.12,Ingenieurbüros für technische Fachplanung und ...,0.004012,0.016403,0.371265,0.435072,0.206310,0.253690,3.036773,3212
27,28.25,Herstellung von kälte- und lufttechnischen Erz...,0.006520,0.053546,0.472760,0.354723,0.198793,0.244163,6.971419,2935
19,26.11,Herstellung von sonstigen elektronischen Bauel...,0.025928,0.040086,0.530612,0.280893,0.164598,0.183548,1.542892,2540
39,45.19,Handel mit Kraftwagen mit einem Gesamtgewicht ...,0.004476,0.027634,0.299960,0.567726,0.076830,0.156855,4.750371,2433
38,45.11,Handel mit Kraftwagen mit einem Gesamtgewicht ...,0.007112,0.032608,0.336695,0.477356,0.074731,0.158806,3.896820,2145
31,28.99,Herstellung von Maschinen für sonstige bestimm...,0.015118,0.026800,0.431319,0.413673,0.161942,0.202705,1.807705,2011


In [116]:
readiness_summary_path = output_folder / "part8_readiness_summary.csv"
final_ratio_availability_path = output_folder / "part8_final_ratio_availability.csv"
sector_ratio_table_path = output_folder / "part8_sector_ratio_table.csv"
status_ratio_table_path = output_folder / "part8_status_ratio_table.csv"
financial_sensitivity_path = output_folder / "part8_financial_sector_sensitivity.csv"
year_ratio_table_path = output_folder / "part8_year_ratio_table.csv"
wz4_ratio_table_path = output_folder / "part8_wz4_ratio_table.csv"

readiness_summary.to_csv(readiness_summary_path, index=False, encoding="utf-8-sig")
ratio_final_quality_df.to_csv(final_ratio_availability_path, index=False, encoding="utf-8-sig")
sector_ratio_table.to_csv(sector_ratio_table_path, index=False, encoding="utf-8-sig")
status_ratio_table.to_csv(status_ratio_table_path, index=False, encoding="utf-8-sig")
financial_sensitivity_summary.to_csv(financial_sensitivity_path, index=False, encoding="utf-8-sig")
year_ratio_table.to_csv(year_ratio_table_path, index=False, encoding="utf-8-sig")
wz4_ratio_table.to_csv(wz4_ratio_table_path, index=False, encoding="utf-8-sig")

print("Saved Part 8 thesis-readiness outputs:")
print(output_folder)

Saved Part 8 thesis-readiness outputs:
/content/drive/MyDrive/IBS_7_Thesis/Python Files/cleaning_audit_outputs
